# hERG (Kv11.1) current analysis by whole-cell patch clamp

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<USERNAME>/<REPOSITORY>/blob/main/analysis_hERG.ipynb)

This notebook implements the procedure for analysing hERG (Kv11.1) currents expressed heterologously in HEK293 cells, from `.abf` recordings acquired with Clampex in the whole-cell configuration. For each file the voltage command is read and the baseline, the test pulse and the tail are detected automatically; a copy of the signal is filtered (zero-phase filter) and the baseline of every sweep is corrected. The following quantities are computed:

| Level | Quantities |
|---|---|
| **Per sweep** | Steady-state current, pulse peak, tail peak, time to peak, conductance and deactivation kinetics |
| **Per cell** | Activation curve (Boltzmann: V½, k, apparent charge z, standard errors and R²), quality control and approximate capacitance |
| **Across cells** | Descriptive statistics, between-group comparisons with correction for multiple comparisons, mean curves, correlations and outliers |

**Notebook format.** The code of each cell is presented folded (form format) and can be expanded for inspection; it contains comments describing the purpose and rationale of each step. Editable parameters are presented as form fields. The statistical unit is the cell; sweeps from the same cell are not treated as independent replicates.

## Structure

| Section | Content |
|---|---|
| **1** | Installation of `pyabf` and library imports |
| **2** | Parameters: (a) filter · (b) protocol, windows and baseline · (c) solution and kinetics · (d) quality control · (e) control cells, capacitances and corrections |
| **3** | Loading of the `.abf` files |
| **4** | Function definitions: (a) single-cell analysis · (b) statistics · (c) plots |
| **5** | Verification of windows, filter and baseline (prior to the analysis) |
| **6** | Analysis of each cell (per-sweep table and nine-panel figure) |
| **7** | Quality control |
| **8–11** | Descriptive statistics, Control vs transfected comparisons, mean curves, Figures 1 and 2, sensitivity of V½ and k, correlations and outliers |
| **12** | Export of tables (source data) |

## Citation

If this code is used, citing the associated article and the repository is requested (see `CITATION.cff`).


## 1. Installation and library imports


In [ ]:
#@title 1a. Install libraries { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: installs pyabf (reads the .abf files generated by Clampex).
# Rationale: Colab does not ship pyabf; it must be installed at the start of each new session.
# ----------------------------------------------------------------------------------------------------------
!pip -q install pyabf

In [ ]:
#@title 1b. Import libraries { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: loads the numerical, plotting and file-handling libraries.
# Rationale: numpy/pandas handle the data; scipy performs the fits (Boltzmann, exponentials), the filters and the statistical tests;
#          matplotlib draws the figures; pyabf reads the recordings. Harmless numerical warnings (e.g. 'covariance') are silenced to keep the output readable.
# ----------------------------------------------------------------------------------------------------------
import os, glob, shutil, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf
from scipy import stats
from scipy.optimize import curve_fit
# butter/bessel design the filters; sosfiltfilt applies them forwards and backwards (zero phase: peaks are not shifted)
from scipy.signal import butter, bessel, sosfiltfilt
from scipy.ndimage import gaussian_filter1d
try:
    from google.colab import files      # only available in Google Colab
except ImportError:
    files = None                        # local execution (Jupyter): the .abf files in the 'data' folder are read
from IPython.display import display

# Divide-by-zero / NaN warnings are expected when a cell has no current: they are not errors
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Covariance.*")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 2. Parameters
The parameters are organised in five blocks: **(a)** filter · **(b)** protocol, windows and baseline · **(c)** solution and kinetics · **(d)** quality control · **(e)** control cells, capacitances and corrections. The default values are those used in the associated study.


In [ ]:
#@title (a) Signal filter { display-mode: "form" }

#@markdown The filter is applied to a **copy** of each trace: to measure peaks, find the tail maximum and plot.
#@markdown Means over flat windows, the baseline, the Cm estimate and the exponential fits use the **unfiltered** data.
#@markdown All filters are **zero-phase**: they do not shift peaks in time.
FILTER_TYPE = "butterworth"  #@param ["butterworth", "gaussian", "bessel", "none"]
CUTOFF_FREQUENCY_HZ = 200.0  #@param {type:"number"}
FILTER_ORDER = 4             #@param {type:"integer"}

#@markdown Only if FILTER_TYPE = "none": moving average used to search for peaks
SMOOTHING_MS = 1.0           #@param {type:"number"}

if FILTER_TYPE == "none" or CUTOFF_FREQUENCY_HZ <= 0:
    FILTER_TYPE = "none"
    FILTER_SIGMA_MS = 0.0
    print(f"No filter (moving average of {SMOOTHING_MS} ms used only to search for peaks)")
else:
    # Approximate time scale of the filter (Gaussian with −3 dB cutoff at fc: sigma = 0.1325 / fc)
    FILTER_SIGMA_MS = 1000 * 0.1325 / CUTOFF_FREQUENCY_HZ
    extra = f", order {FILTER_ORDER}" if FILTER_TYPE == "butterworth" else ""
    print(f"Filter {FILTER_TYPE} at {CUTOFF_FREQUENCY_HZ:g} Hz{extra} (time scale ≈ {FILTER_SIGMA_MS:.2f} ms)")

In [ ]:
#@title (b) Protocol, windows and baseline { display-mode: "form" }

#@markdown **Windows.** In *automatic* mode the notebook reads the command waveform of each file and detects the baseline, the pulse and the tail on its own.
#@markdown In *manual* mode the durations given below are used (defined for the hERG1.pro protocol).
WINDOW_MODE = "automatic"  #@param ["automatic", "manual"]

#@markdown ---
#@markdown **Baseline.** The **median** of the (unfiltered) current in a window is subtracted from the whole sweep:
#@markdown this corrects the offset of *each sweep*. There are two possible places to measure it:
#@markdown - **initial holding**: within the first holding segment of the sweep (window from start to end below).
#@markdown - **before pulse**: the last ms before the test pulse.
#@markdown The notebook checks that the chosen window is really at the reference voltage (−80 mV by default).
#@markdown If it is not, the other option is used and a warning is issued. **Measuring the hERG baseline at 0 mV is not valid**, because hERG current flows at that potential.
BASELINE_MODE = "initial holding"  #@param ["initial holding", "before pulse"]
BASELINE_WINDOW_START_MS = 10.0  #@param {type:"number"}
BASELINE_WINDOW_END_MS = 70.0   #@param {type:"number"}
BASELINE_WINDOW_MS = 100          #@param {type:"number"}
V_BASELINE_REF_mV = -80            #@param {type:"number"}
V_BASELINE_TOL_mV = 5              #@param {type:"number"}

#@markdown ---
#@markdown **Measurement windows**
STEADY_WINDOW_MS = 50      #@param {type:"number"}
TAIL_PEAK_WINDOW_MS = 500   #@param {type:"number"}
SKIP_TRANSIENT_MS = 3.0   #@param {type:"number"}
PULSE_BLANKING_MS = 30          #@param {type:"number"}
TAIL_BLANKING_MS = 5            #@param {type:"number"}
CHANNEL = 0                    #@param {type:"integer"}

#@markdown ---
#@markdown **Epoch durations (manual mode only)**
INITIAL_HOLDING_MS = 81.2    #@param {type:"number"}
EPOCH_A_MS = 500             #@param {type:"number"}
EPOCH_B_MS = 2000            #@param {type:"number"}
EPOCH_C_MS = 2000            #@param {type:"number"}
EPOCH_D_MS = 500             #@param {type:"number"}

# Margin around each voltage step: the filter "smears" the capacitive transient by about 4 sigma.
JUMP_MARGIN_MS = max(SKIP_TRANSIENT_MS, 4 * FILTER_SIGMA_MS)
# After the step to the pulse and to the tail there is a transient that lasts longer than the pure capacitive one: more time is skipped.
PULSE_BLANKING_MS = max(PULSE_BLANKING_MS, JUMP_MARGIN_MS)
TAIL_BLANKING_MS = max(TAIL_BLANKING_MS, JUMP_MARGIN_MS)
print(f"Window mode: {WINDOW_MODE} · preferred baseline: '{BASELINE_MODE}' at {V_BASELINE_REF_mV:g} mV (±{V_BASELINE_TOL_mV:g})")
print(f"Margins: voltage steps {JUMP_MARGIN_MS:.1f} ms · pulse onset {PULSE_BLANKING_MS:.0f} ms · tail onset {TAIL_BLANKING_MS:.0f} ms")
print("The exact windows are computed per file and shown in section 5.")

In [ ]:
#@title (c) Solution, temperature and kinetics { display-mode: "form" }

#@markdown **K⁺ equilibrium potential (Nernst)** — must match the composition of the solutions used
K_INT_mM = 130        #@param {type:"number"}
K_EXT_mM = 4.0        #@param {type:"number"}
TEMPERATURE_C = 22.0  #@param {type:"number"}

#@markdown **Kinetics**
V_REF_mV = 40         #@param {type:"number"}
SNR_THRESHOLD = 4        #@param {type:"number"}
MIN_THRESHOLD_pA = 10    #@param {type:"number"}

R_GAS = 8.314462618
F_CONST = 96485.33212
RT_F_mV = 1000 * R_GAS * (273.15 + TEMPERATURE_C) / F_CONST
EK_mV = RT_F_mV * np.log(K_EXT_mM / K_INT_mM)
print(f"E_K (Nernst) = {EK_mV:.1f} mV   |   RT/F = {RT_F_mV:.2f} mV")
print(f"A current is considered genuine if it exceeds {SNR_THRESHOLD}× the (filtered) baseline noise and at least {MIN_THRESHOLD_pA} pA.")

In [ ]:
#@title (d) Quality control { display-mode: "form" }
QC_MAX_HOLDING_I_pA = 200           #@param {type:"number"}
QC_MAX_NOISE_pA = 30             #@param {type:"number"}
QC_MAX_DRIFT_pA_PER_SWEEP = 10  #@param {type:"number"}
USE_ESTIMATED_CM = False         #@param {type:"boolean"}

In [ ]:
#@title (e) Control cells, capacitances and corrections { display-mode: "form" }
import re

# ----------------------------------------------------------------------------------------------------------
# Description: defines (1) which files belong to the CONTROL group, (2) the capacitance of each cell and
#           (3) whether the leak and the offset of the current are corrected.
# Rationale: the Control vs transfected comparison needs to know which group each cell belongs to, and converting pA to
#          pA/pF (current density) requires dividing by the Cm of EACH cell. They are entered as form text
#          so that the code can stay folded when the notebook is read.
# ----------------------------------------------------------------------------------------------------------
#@markdown **Controls.** Comma-separated names of the .abf files of the control cells, e.g. `recording_01.abf, recording_02.abf`. All other files are assigned to the transfected group. Leave empty if there are no controls.
CONTROL_FILES = ""  #@param {type:"string"}
CONTROL_GROUP = "Control"  #@param {type:"string"}
TRANSFECTED_GROUP = "hERG"       #@param {type:"string"}

#@markdown **Measured capacitances (pF)** (optional), format `file: value` separated by `;` or commas, e.g. `recording_03.abf: 20.5; recording_04.abf: 15.2`.
#@markdown Note: capacitances should be entered for ALL cells (controls included). If any is missing, figures and tests that compare groups are expressed in pA so that units are not mixed (the notebook issues a warning).
CAPACITANCES_pF = ""  #@param {type:"string"}

#@markdown **OPTIONAL current corrections**
#@markdown - **LEAK_CORRECTION = holding**: subtracts from the WHOLE sweep a linear leak g·(V − V_base), with g = I_holding / (V_base − E_FUGA). It corrects only the leak visible in the holding current and assumes that it reverses at E_FUGA.
#@markdown - **OFFSET_CORRECTION = V_min**: subtracts from every tail the tail measured at the most negative voltage (−70 mV), where hERG is assumed not to be activated. The original tail is kept in `I_tail_raw_pA`.
LEAK_CORRECTION = "holding"   #@param ["none", "holding"]
E_LEAK_mV = 0.0              #@param {type:"number"}
OFFSET_CORRECTION = "V_min"    #@param ["none", "V_min"]

def _stem(name):
    """'26o01001.abf' -> '26o01001' (so it does not matter whether the extension is typed)."""
    name = name.strip()
    return name[:-4] if name.lower().endswith(".abf") else name

# Text -> dictionaries used by the rest of the notebook (keys carry .abf, like the loaded files)
GROUPS = {_stem(t) + ".abf": CONTROL_GROUP for t in re.split(r"[;,\n]+", CONTROL_FILES) if t.strip()}
CM_PF = {_stem(a) + ".abf": float(v.replace(",", "."))
         for a, v in re.findall(r"([\w\-]+(?:\.abf)?)\s*[:=]\s*([0-9]+(?:[.,][0-9]+)?)", CAPACITANCES_pF)}

def group_of(name):
    """Group of a file: Control if listed in CONTROL_FILES; otherwise the transfected group (TRANSFECTED_GROUP)."""
    return GROUPS.get(_stem(name) + ".abf", TRANSFECTED_GROUP)

def order_groups(gs):
    """Control first, then transfected, then any other group (so that figures always appear in the same order)."""
    gs = list(gs)
    return [g for g in (CONTROL_GROUP, TRANSFECTED_GROUP) if g in gs] + [g for g in gs if g not in (CONTROL_GROUP, TRANSFECTED_GROUP)]

print(f"Control files ({len(GROUPS)}): {', '.join(sorted(GROUPS))}")
print(f"Capacitances read: {len(CM_PF)} cells")
print(f"Corrections → leak: {LEAK_CORRECTION} (E = {E_LEAK_mV:g} mV) · tail offset: {OFFSET_CORRECTION}")


## 3. Loading `.abf` files
One or several recordings (one per cell) are loaded. In Google Colab a file selector opens; in local execution the files present in the `data/` folder are read.


In [ ]:
#@title 3. Load .abf files { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: gathers the .abf recordings in the 'data' folder and lists the group assigned to each file.
# Rationale: each .abf corresponds to one cell; the group (Control / transfected) is assigned from the file name (section 2e).
#   - Google Colab: the file selector opens and the uploaded recordings are moved to 'data'.
#   - Local execution: the .abf files already present in the 'data' folder are read directly.
# ----------------------------------------------------------------------------------------------------------
os.makedirs("data", exist_ok=True)
if files is not None:
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith(".abf"):
            shutil.move(name, os.path.join("data", name))
abf_files = sorted(glob.glob("data/*.abf"))
print(f"{len(abf_files)} file(s) available:")
for a in abf_files:
    print("  ·", os.path.basename(a), "→ group:", group_of(os.path.basename(a)))


## 4. Functions
This section only defines functions; no computation is performed.

**4a. Single-cell analysis** · **4b. Statistics** · **4c. Plots**


### 4a. Single-cell analysis

In [ ]:
#@title 4a. Single-cell analysis functions { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: defines the tools to analyse ONE recording: windows, filter, baseline, peaks, fits and quality control.
# Rationale: separating 'defining' from 'running' allows the method to be reviewed without computing anything and to be reused identically for every cell.
# Nothing is computed here yet. Summary of the method for each sweep (one sweep = one test voltage):
#   1) baseline = median of the UNFILTERED current at −80 mV (corrects the offset of that sweep)
#   2) steady state = mean at the end of the pulse · pulse peak and tail peak = maxima of the FILTERED signal
#   3) tail normalised to the cell maximum → activation curve → Boltzmann fit
#   4) deactivation = exponential fit of the tail (unfiltered data)
# ----------------------------------------------------------------------------------------------------------
# np.trapz is called np.trapezoid in numpy ≥ 2.0: whichever exists is used so that the notebook works with any version
_integra = getattr(np, "trapezoid", None) or np.trapz

# Boolean mask: True for the samples with start_ ≤ t < stop_. Every "window" in the notebook is computed this way.
def window_mask(t, start_, stop_):
    return (t >= start_) & (t < stop_)

# Moving average. Only used when FILTER_TYPE = "none", so that the maximum found is not an isolated noise spike.
def smooth(y, fs, ms):
    n = max(1, int(fs * ms / 1000))
    return np.convolve(y, np.ones(n) / n, mode="same")

# Averages blocks of points (≈1500 final points): greatly speeds up the exponential fits without changing the shape of the curve.
def bin_signal(t, y, max_pts=1500):
    """Averages blocks of points to speed up the fits without losing the shape."""
    step = max(1, len(t) // max_pts)
    m = (len(t) // step) * step
    return t[:m].reshape(-1, step).mean(1), y[:m].reshape(-1, step).mean(1)

# ---------- Models ----------
# Activation curve: a / (1 + exp((V½ − V)/k)). V½ = half-activation voltage; k = slope (mV); a = maximal amplitude (≈1 once normalised).
def boltzmann(v, a, v50, k):
    return a / (1.0 + np.exp((v50 - v) / k))

# Deactivation with ONE exponential: a·exp(−t/τ) + c (c = residual current at the end).
def model_monoexp(t, a, tau, c):
    return a * np.exp(-t / tau) + c

# Deactivation with TWO exponentials (fast and slow component), typical of hERG.
def model_biexp(t, a1, tau1, a2, tau2, c):
    return a1 * np.exp(-t / tau1) + a2 * np.exp(-t / tau2) + c

# ---------- Fits ----------
# Fits a Boltzmann and returns V½, k, their standard errors, R² and the apparent charge z = RT/(F·k).
def fit_boltzmann(v, y):
    """Boltzmann with standard errors, R² and apparent charge z = RT/(F·k)."""
    v, y = np.asarray(v, float), np.asarray(y, float)
    ok = np.isfinite(v) & np.isfinite(y)
    # With fewer than 4 valid points a 3-parameter fit is meaningless: None is returned
    if ok.sum() < 4:
        return None
    v, y = v[ok], y[ok]
    try:
        # The bounds prevent absurd solutions: amplitude 0.5–1.5 (the curve is normalised), V½ between −100 and +100 mV, k between 1 and 60 mV.
        # If the fit sticks to a bound, that V½/k is NOT well determined (flagged in section 10e).
        popt, pcov = curve_fit(boltzmann, v, y, p0=[1.0, np.median(v), 10.0],
                               bounds=([0.5, -100, 1.0], [1.5, 100, 60]), maxfev=20000)
    except Exception:
        return None
    # Standard error of each parameter = square root of the diagonal of the covariance matrix returned by curve_fit
    se = np.sqrt(np.diag(pcov))
    ss_res = np.sum((y - boltzmann(v, *popt)) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return {"A": popt[0], "V50_mV": popt[1], "k_mV": popt[2],
            "SE_A": se[0], "SE_V50": se[1], "SE_k": se[2],
            "R2": 1 - ss_res / ss_tot if ss_tot > 0 else np.nan,
            "z_apparent": RT_F_mV / popt[2]}

# Fits mono- and bi-exponential models to the tail. The bi-exponential is chosen only if it is clearly better (AIC criterion), to avoid overfitting.
def fit_deactivation(t, y):
    """Fits mono- and bi-exponential models to the tail (t in ms from the peak, y in pA).
    - The bi-exponential is chosen only if it improves the AIC by more than 10 units.
    - If τ is larger than 3 times the fitted duration, it CANNOT be determined with that window
      (the curve is almost a straight line) and NaN is returned instead of a meaningless value."""
    n = len(t)
    dur = t[-1] - t[0]
    # Rule of thumb: a τ much longer than the fitted window cannot be measured (the curve looks like a straight line).
    tau_max_reliable = 3 * dur
    a0, c0 = max(y[0] - y[-1], 1e-6), y[-1]
    try:
        p1, _ = curve_fit(model_monoexp, t, y, p0=[a0, min(200.0, dur / 2), c0],
                          bounds=([0, 1.0, -np.inf], [np.inf, 20000.0, np.inf]), maxfev=20000)
    except Exception:
        return None
    rss1 = np.sum((y - model_monoexp(t, *p1)) ** 2)
    out = {"tau_mono_ms": p1[1], "p_mono": tuple(p1), "model": "mono-exp",
           "tau_fast_ms": np.nan, "tau_slow_ms": np.nan, "fast_fraction": np.nan}
    # If τ is more than 3 times the window, NaN is returned instead of a meaningless value
    if p1[1] > tau_max_reliable:
        out["tau_mono_ms"] = np.nan
        out["model"] = "τ not determinable (short window)"
        return out
    # τ stuck at the lower bound (1 ms) = the fit chased the capacitive transient, not a real exponential
    if p1[1] < 1.5:                      # stuck at the lower bound of the fit: not a real exponential
        out["tau_mono_ms"] = np.nan
        out["model"] = "τ not determinable (fit at bound)"
        return out
    try:
        p2, _ = curve_fit(model_biexp, t, y, p0=[0.5 * a0, 30.0, 0.5 * a0, min(400.0, dur / 2), c0],
                          bounds=([0, 1.0, 0, 1.0, -np.inf], [np.inf, 20000.0, np.inf, 20000.0, np.inf]),
                          maxfev=40000)
        rss2 = np.sum((y - model_biexp(t, *p2)) ** 2)
        # AIC = n·ln(RSS/n) + 2·(parameters): penalises models with more parameters. The bi-exp wins only if it improves the AIC by more than 10 units.
        aic1 = n * np.log(max(rss1, 1e-12) / n) + 2 * 3
        aic2 = n * np.log(max(rss2, 1e-12) / n) + 2 * 5
        a1, t1, a2, t2, c = p2
        if t1 > t2:
            a1, t1, a2, t2 = a2, t2, a1, t1
        out["p_bi"] = (a1, t1, a2, t2, c)
        if aic2 < aic1 - 10 and t2 <= tau_max_reliable:
            out.update(model="bi-exp", tau_fast_ms=t1, tau_slow_ms=t2, fast_fraction=a1 / (a1 + a2))
    except Exception:
        pass
    return out

# ---------- Filtro ----------
def apply_filter(y, fs):
    """ZERO-PHASE low-pass filter (does not shift peaks in time).
    With zero phase the filter is applied forwards and backwards, so the effective (−3 dB) cutoff
    lies somewhat below CUTOFF_FREQUENCY_HZ."""
    y = np.asarray(y, float)
    if FILTER_TYPE == "none":
        return y.copy()
    if CUTOFF_FREQUENCY_HZ >= fs / 2:
        raise ValueError(f"CUTOFF_FREQUENCY_HZ ({CUTOFF_FREQUENCY_HZ}) must be lower than half the sampling frequency ({fs/2:.0f} Hz)")
    if FILTER_TYPE == "butterworth":
        sos = butter(FILTER_ORDER, CUTOFF_FREQUENCY_HZ, btype="low", fs=fs, output="sos")
        return sosfiltfilt(sos, y)
    if FILTER_TYPE == "gaussian":
        return gaussian_filter1d(y, sigma=0.1325 / CUTOFF_FREQUENCY_HZ * fs, mode="nearest")
    if FILTER_TYPE == "bessel":
        sos = bessel(8, CUTOFF_FREQUENCY_HZ, btype="low", fs=fs, output="sos", norm="mag")
        return sosfiltfilt(sos, y)
    raise ValueError(f"Unknown FILTER_TYPE: {FILTER_TYPE}")

# Signal on which peaks are SEARCHED: the filtered one (or a moving average if there is no filter). MEANS over flat windows use the raw data.
def analysis_signal(y, fs):
    """Signal used to measure peaks: filtered, or a moving average if the filter is off."""
    return apply_filter(y, fs) if FILTER_TYPE != "none" else smooth(y, fs, SMOOTHING_MS)

# "Genuine current" criterion: it must exceed SNR times the baseline noise AND an absolute minimum in pA. Avoids measuring noise as if it were signal.
def signal_threshold(noise):
    """An amplitude counts as a genuine current if it exceeds SNR × noise and an absolute minimum."""
    return max(SNR_THRESHOLD * noise, MIN_THRESHOLD_pA)

# ---------- Measurement windows and baseline (computed per file) ----------
# Computes the windows (baseline, pulse, steady state, tail) of THIS file and stores them in global variables.
# The LAST sweep (the most depolarised) is used because the command shows all the segments of the protocol there.
def apply_windows(abf):
    """Computes the windows of this file and stores them in global variables:
    BASELINE_WIN, BASELINE_USED, BASELINE_ALERT, BASELINE_FAILED, V_BASELINE, V_PRE, PULSE_WIN, STEADY_WIN, TAIL_WIN,
    TAIL_PEAK_SEARCH_WIN, PULSE_PEAK_SEARCH_WIN, V_TAIL."""
    global BASELINE_WIN, BASELINE_USED, BASELINE_ALERT, BASELINE_FAILED, V_BASELINE, V_PRE, V_TAIL
    global PULSE_WIN, STEADY_WIN, TAIL_WIN, TAIL_PEAK_SEARCH_WIN, PULSE_PEAK_SEARCH_WIN
    fs = abf.dataRate
    abf.setSweep(abf.sweepCount - 1, channel=CHANNEL)      # last sweep = the most depolarised
    t, cmd = abf.sweepX, abf.sweepC
    if WINDOW_MODE == "manual":
        a_end = (INITIAL_HOLDING_MS + EPOCH_A_MS) / 1000
        b_end = a_end + EPOCH_B_MS / 1000
        c_end = b_end + EPOCH_C_MS / 1000
        start_seg, baseline_seg = (0.0, INITIAL_HOLDING_MS / 1000), (0.0, a_end)
        pulse, tail = (a_end, b_end), (b_end, c_end)
    else:
        # The command is piecewise constant: the steps are located and the segments are built.
        # The command is piecewise constant (steps): each voltage jump (>0.5 mV) marks the start of a new segment
        changes = np.where(np.abs(np.diff(cmd)) > 0.5)[0] + 1
        edges_ = np.concatenate([[0], changes, [len(cmd)]])
        segs = [(t[a], t[b - 1] + 1 / fs, float(np.median(cmd[a:b])))
                for a, b in zip(edges_[:-1], edges_[1:]) if (b - a) / fs >= 0.005]
        segment_list = "; ".join(f"{sg[0]:.3f}–{sg[1]:.3f} s at {sg[2]:.0f} mV" for sg in segs)

        # 1) The test pulse is the segment whose voltage CHANGES between sweeps (first sweep vs. last).
        k = None
        if abf.sweepCount > 1:
            abf.setSweep(0, channel=CHANNEL)
            cmd_first = np.array(abf.sweepC)
            abf.setSweep(abf.sweepCount - 1, channel=CHANNEL)
            diff_ = np.abs(cmd - cmd_first) > 0.5
            counts = [int(diff_[(t >= sg[0]) & (t < sg[1])].sum()) for sg in segs]
            if counts and max(counts) > 0:
                k = int(np.argmax(counts))
        # 2) If the sweeps are identical: the longest segment that is neither the first nor the last.
        if k is None:
            interior = [j for j in range(1, len(segs) - 1) if segs[j][1] - segs[j][0] >= 0.1]
            if interior:
                k = max(interior, key=lambda j: segs[j][1] - segs[j][0])
        if k is None or k == 0 or k == len(segs) - 1:
            raise ValueError("The test pulse could not be identified (a segment before and another after it are needed). "
                             f"Command segments: {segment_list}. WINDOW_MODE = 'manual' must be used.")
        start_seg, baseline_seg, pulse, tail = segs[0][:2], segs[k - 1][:2], segs[k][:2], segs[k + 1][:2]

    # m = margin around each voltage step, where the capacitive artefact (and the "smearing" of the filter) is ignored
    m = JUMP_MARGIN_MS / 1000
    V_TAIL = float(np.median(cmd[window_mask(t, tail[0] + 0.01, tail[1] - 0.01)]))
    PULSE_WIN, TAIL_WIN = tuple(pulse), tuple(tail)
    # Steady state = last STEADY_WINDOW_MS of the pulse (the current no longer changes); a margin is left before the next step
    STEADY_WIN = (max(pulse[0], pulse[1] - STEADY_WINDOW_MS / 1000 - m), pulse[1] - m)
    PULSE_PEAK_SEARCH_WIN = (pulse[0] + PULSE_BLANKING_MS / 1000, pulse[1] - m)
    # The tail peak is searched only in the first TAIL_PEAK_WINDOW_MS after the step, skipping the initial capacitive artefact
    TAIL_PEAK_SEARCH_WIN = (tail[0] + TAIL_BLANKING_MS / 1000, min(tail[1] - m, tail[0] + TAIL_PEAK_WINDOW_MS / 1000))

    # --- Baseline: two candidates ---
    # Command voltage (median) in a window: used to check that the baseline is really at −80 mV
    def level_at(w):
        mk = window_mask(t, *w)
        return float(np.median(cmd[mk])) if mk.any() else np.nan
    start_ = (BASELINE_WINDOW_START_MS / 1000, min(BASELINE_WINDOW_END_MS / 1000, start_seg[1] - m))          # initial holding
    pre = (max(baseline_seg[0] + m, baseline_seg[1] - BASELINE_WINDOW_MS / 1000 - m), baseline_seg[1] - m)   # just before the pulse
    cand = {"initial holding": (start_, level_at(start_)), "before pulse": (pre, level_at(pre))}
    V_PRE = cand["before pulse"][1]
    order_ = ["before pulse", "initial holding"] if BASELINE_MODE == "before pulse" else ["initial holding", "before pulse"]

    # A baseline window is valid if its command is at V_BASE_REF ± tolerance. At another voltage (e.g. 0 mV) hERG current flows and it is NOT a baseline.
    def is_valid(nm):
        w, v = cand[nm]
        return np.isfinite(v) and w[1] > w[0] and abs(v - V_BASELINE_REF_mV) <= V_BASELINE_TOL_mV
    chosen_one = next((nm for nm in order_ if is_valid(nm)), None)
    BASELINE_ALERT, BASELINE_FAILED = "", False
    if chosen_one is None:
        chosen_one, BASELINE_FAILED = order_[0], True
        BASELINE_ALERT = (f"No baseline window is at {V_BASELINE_REF_mV:.0f} mV "
                       f"(initial holding: {cand['initial holding'][1]:.0f} mV; before pulse: {cand['before pulse'][1]:.0f} mV). "
                       f"Current at any other voltage is NOT a valid hERG baseline: check V_BASELINE_REF_mV.")
    elif chosen_one != order_[0]:
        BASELINE_ALERT = (f"'{order_[0]}' is at {cand[order_[0]][1]:.0f} mV (not at {V_BASELINE_REF_mV:.0f} mV): "
                       f"'{chosen_one}' ({cand[chosen_one][1]:.0f} mV) was used.")
    BASELINE_WIN, V_BASELINE = cand[chosen_one]
    BASELINE_USED = chosen_one

    # --- Linear leak estimated from the holding current (only if LEAK_CORRECTION = "holding") ---
    # Linear leak: g = I_holding / (V_base − E_FUGA). g·(V − V_base) is subtracted from the whole sweep. It only corrects the part of the leak visible at holding.
    abf._g_leak_nS = 0.0
    if LEAK_CORRECTION == "holding" and abs(V_BASELINE - E_LEAK_mV) > 1:
        ih = []
        for s_ in abf.sweepList:
            abf.setSweep(s_, channel=CHANNEL)
            ih.append(np.median(np.array(abf.sweepY)[window_mask(np.array(abf.sweepX), *BASELINE_WIN)]))
        abf._g_leak_nS = float(np.mean(ih)) / (V_BASELINE - E_LEAK_mV)        # pA/mV = nS
        abf.setSweep(abf.sweepCount - 1, channel=CHANNEL)

# ---------- Approximate capacitance ----------
# Raw (UNFILTERED) current of the current sweep (with the linear leak subtracted if enabled). It is the basis of all measurements.
def raw_current(abf):
    """Unfiltered current of the current sweep. With LEAK_CORRECTION = "holding" the linear leak g·(V − V_base) is subtracted
    (g comes from apply_windows, which must have been called beforehand for this file)."""
    i = np.array(abf.sweepY, dtype=float)
    g = getattr(abf, "_g_leak_nS", 0.0)
    if LEAK_CORRECTION == "holding" and g:
        i = i - g * (np.array(abf.sweepC, dtype=float) - V_BASELINE)
    return i

# Cm ≈ charge of the capacitive transient / ΔV. It is approximate: if Cm or Rs were compensated at the amplifier, it is underestimated. Hence the measured Cm is preferred (section 2e).
def estimate_cm(abf):
    """Cm ≈ charge of the capacitive transient / ΔV, at the smallest-amplitude step.
    Approximate: underestimates if Cm or Rs were compensated at the amplifier."""
    apply_windows(abf)
    best = None
    for s in abf.sweepList:
        abf.setSweep(s, channel=CHANNEL)
        t, i, cmd = abf.sweepX, abf.sweepY, abf.sweepC
        dv = np.mean(cmd[window_mask(t, PULSE_WIN[0] + 0.1, PULSE_WIN[1] - 0.1)]) - V_PRE
        if abs(dv) < 2:
            continue
        if best is None or abs(dv) < abs(best[0]):
            best = (dv, np.array(t), np.array(i))
    if best is None:
        return np.nan
    dv, t, i = best
    w = window_mask(t, PULSE_WIN[0], PULSE_WIN[0] + 0.020)
    steady = np.mean(i[window_mask(t, PULSE_WIN[0] + 0.040, PULSE_WIN[0] + 0.060)])
    q_pC = _integra(i[w] - steady, t[w])          # pA·s = pC
    return abs(q_pC / dv) * 1000                   # pC/mV = nF → pF

# ---------- Single-cell analysis ----------
# Complete analysis of ONE cell: loops over every sweep, measures, fits and returns table + Boltzmann + quality control.
def analyze_abf(path_, cm_pF=None):
    abf = pyabf.ABF(path_)
    fs = abf.dataRate
    apply_windows(abf)
    baseline_info = {"window_mask": BASELINE_USED, "start_s": BASELINE_WIN[0], "end_s": BASELINE_WIN[1], "V_mV": V_BASELINE,
                 "alert": BASELINE_ALERT, "failed": BASELINE_FAILED}
    cm_est = estimate_cm(abf)          # uses unfiltered data
    if cm_pF:
        cm_used, cm_source = cm_pF, "manual"
    elif USE_ESTIMATED_CM and np.isfinite(cm_est):
        cm_used, cm_source = cm_est, "estimated"
    else:
        cm_used, cm_source = None, "not used"

    # Preliminary pass: reference time of the tail peak (median over the sweeps with a clear current)
    t_peaks = []
    for s in abf.sweepList:
        abf.setSweep(s, channel=CHANNEL)
        t, i = abf.sweepX, raw_current(abf)
        y = analysis_signal(i, fs)
        m_b = window_mask(t, *BASELINE_WIN)
        idx_c = np.where(window_mask(t, *TAIL_PEAK_SEARCH_WIN))[0]
        if len(idx_c) == 0:
            continue
        kc = idx_c[np.argmax(y[idx_c])]
        if y[kc] - np.median(i[m_b]) > signal_threshold(np.std(y[m_b])):
            t_peaks.append(t[kc])
    t_ref = np.median(t_peaks) if t_peaks else np.nan

    # Main loop: one row of results per sweep. "example" keeps the largest tail (used to show the deactivation fit).
    rows_, example, best_tail = [], None, -np.inf
    for s in abf.sweepList:
        abf.setSweep(s, channel=CHANNEL)
        t, i, cmd = abf.sweepX, raw_current(abf), abf.sweepC
        y = analysis_signal(i, fs)

        # --- baseline of THIS sweep: median of the unfiltered current in the chosen window ---
        m_base = window_mask(t, *BASELINE_WIN)
        if not m_base.any():
            raise ValueError(f"The baseline window {BASELINE_WIN} contains no samples in sweep {s}.")
        # BASELINE OF THIS SWEEP: median (robust to outliers) of the unfiltered current. It is subtracted from every measurement of this sweep.
        baseline = np.median(i[m_base])
        # Noise = standard deviation of the baseline. The noise of the FILTERED signal decides whether a current is genuine (below)
        noise_raw = np.std(i[m_base])             # noise of the unfiltered signal
        noise = np.std(y[m_base])                 # noise of the filtered signal: decides whether a current is genuine
        threshold = signal_threshold(noise)

        v_test = np.mean(cmd[window_mask(t, PULSE_WIN[0] + 0.1, PULSE_WIN[1] - 0.1)])
        # Steady-state current (end of the pulse) = mean over a flat window, unfiltered, minus the baseline
        i_ss = np.mean(i[window_mask(t, *STEADY_WIN)]) - baseline

        # pulse peak (the transient that follows the voltage step is skipped)
        # Pulse peak: maximum of the filtered signal, skipping the transient that follows the voltage step
        idx_p = np.where(window_mask(t, *PULSE_PEAK_SEARCH_WIN))[0]
        i_pp, t_pp = np.nan, np.nan
        if len(idx_p):
            kp = idx_p[np.argmax(y[idx_p])]
            if y[kp] - baseline > threshold:
                i_pp, t_pp = y[kp] - baseline, (t[kp] - PULSE_WIN[0]) * 1000

        # tail peak
        idx_c = np.where(window_mask(t, *TAIL_PEAK_SEARCH_WIN))[0]
        kc = idx_c[np.argmax(y[idx_c])]
        i_tail = y[kc] - baseline
        t_tail = (t[kc] - TAIL_WIN[0]) * 1000
        # If the peak does not exceed the threshold, the "maximum" would be only noise and would bias the measurement upwards
        with_signal = i_tail > threshold
        method = "peak"
        # No clear signal: measured in a fixed window (±2 ms) around the cell's typical peak time, instead of looking for the noise maximum
        if not with_signal and np.isfinite(t_ref):
            method = "window_mask"
            # Small current: the noise maximum biases upwards. It is measured in a
            # fixed window (±2 ms) around the reference peak time.
            i_tail = np.mean(y[window_mask(t, t_ref - 0.002, t_ref + 0.002)]) - baseline
            t_tail = np.nan

        row_ = {"sweep": s, "V_test_mV": round(v_test, 1),
                "I_baseline_pA": baseline, "noise_pA": noise_raw, "noise_filtered_pA": noise,
                "I_steady_pA": i_ss, "I_pulse_peak_pA": i_pp, "t_pulse_peak_ms": t_pp,
                "I_tail_pA": i_tail, "tail_method": method, "t_tail_peak_ms": t_tail,
                "tau_mono_ms": np.nan, "tau_fast_ms": np.nan, "tau_slow_ms": np.nan,
                "fast_fraction": np.nan, "deact_model": ""}

        # deactivation: from the tail peak to the end of the epoch (unfiltered data, in blocks)
        if with_signal:
            seg = window_mask(t, t[kc], TAIL_WIN[1])
            tt, yy = bin_signal((t[seg] - t[kc]) * 1000, i[seg] - baseline)
            fit = fit_deactivation(tt, yy)
            if fit:
                row_.update(tau_mono_ms=fit["tau_mono_ms"], tau_fast_ms=fit["tau_fast_ms"],
                            tau_slow_ms=fit["tau_slow_ms"], fast_fraction=fit["fast_fraction"],
                            deact_model=fit["model"])
                if i_tail > best_tail:
                    best_tail = i_tail
                    example = {"t": tt, "y": yy, "fit": fit, "V": v_test, "sweep": s}
        rows_.append(row_)

    df = pd.DataFrame(rows_)
    # The original tail is stored before any offset correction (so that the correction can be audited)
    df["I_tail_raw_pA"] = df["I_tail_pA"]
    tail_offset_val, offset_warning = 0.0, ""
    # Offset correction: the tail at the most negative voltage (−70 mV) should be ≈ 0 (hERG closed); whatever is there is foreign constant current and is subtracted from all tails
    if OFFSET_CORRECTION == "V_min":
        tail_offset_val = float(df.loc[df["V_test_mV"].idxmin(), "I_tail_pA"])     # tail at the most negative voltage
        df["I_tail_pA"] = df["I_tail_pA"] - tail_offset_val
        met_ = df["tail_method"]
        if met_.nunique() > 1 and met_.loc[df["V_test_mV"].idxmin()] != met_.mode().iloc[0]:
            offset_warning = ("the minimum-voltage sweep was measured differently (fixed window) from the rest (peak): "
                            "the subtracted offset is not comparable; reviewing this cell is recommended.")
    # Tail normalised to the maximum of THIS cell (0–1): this is the activation curve fitted with Boltzmann
    df["tail_norm"] = df["I_tail_pA"] / df["I_tail_pA"].max() if df["I_tail_pA"].max() > 0 else np.nan
    # Conductance G = I / (V − E_K). Omitted near E_K (|V − E_K| < 5 mV) because the division blows up there
    dv = df["V_test_mV"] - EK_mV
    df["G_steady_nS"] = np.where(np.abs(dv) > 5, df["I_steady_pA"] / dv, np.nan)   # pA/mV = nS
    # Current density: pA divided by the capacitance (pF) → pA/pF, which allows cells of different size to be compared
    if cm_used:
        for col in ["I_steady", "I_pulse_peak", "I_tail"]:
            df[f"{col}_pApF"] = df[f"{col}_pA"] / cm_used
        df["G_steady_pSpF"] = df["G_steady_nS"] * 1000 / cm_used

    # Fit of the activation curve of this cell (V½, k, R²)
    boltz = fit_boltzmann(df["V_test_mV"], df["tail_norm"])

    # quality control
    # Drift = slope of the baseline across sweeps (pA per sweep): if it grows, the seal or the cell is deteriorating
    drift = np.polyfit(df["sweep"], df["I_baseline_pA"], 1)[0] if len(df) > 2 else np.nan
    qc = {"I_hold_mean_pA": df["I_baseline_pA"].mean(), "noise_mean_pA": df["noise_filtered_pA"].mean(),
          "drift_pA_per_sweep": drift}
    # Quality control: every failed criterion adds an alert; no alerts = "OK" (thresholds in section 2d)
    alerts = []
    if baseline_info["failed"]:                            alerts.append("baseline at an incorrect voltage")
    if abs(qc["I_hold_mean_pA"]) > QC_MAX_HOLDING_I_pA: alerts.append("high baseline current (leak)")
    if qc["noise_mean_pA"] > QC_MAX_NOISE_pA:        alerts.append("high noise")
    if abs(drift) > QC_MAX_DRIFT_pA_PER_SWEEP:      alerts.append("baseline drift")
    if boltz is None:                                 alerts.append("Boltzmann did not converge")
    elif boltz["R2"] < 0.95:                          alerts.append("poor Boltzmann fit (R²<0.95)")
    qc["status"] = "OK" if not alerts else "; ".join(alerts)

    return {"df": df, "boltz": boltz, "cm_est": cm_est, "cm_used": cm_used, "cm_source": cm_source,
            "example": example, "qc": qc, "baseline_info": baseline_info,
            "g_leak_nS": getattr(abf, "_g_leak_nS", 0.0), "tail_offset_pA": tail_offset_val, "offset_warning": offset_warning}

def _window_summary():
    print(f"  baseline ('{BASELINE_USED}'): {BASELINE_WIN[0]:.3f}–{BASELINE_WIN[1]:.3f} s, command at {V_BASELINE:.0f} mV")
    print(f"  pulse: {PULSE_WIN[0]:.3f}–{PULSE_WIN[1]:.3f} s · steady state: {STEADY_WIN[0]:.3f}–{STEADY_WIN[1]:.3f} s")
    print(f"  tail: {TAIL_WIN[0]:.3f}–{TAIL_WIN[1]:.3f} s, command at {V_TAIL:.0f} mV · peak searched: {TAIL_PEAK_SEARCH_WIN[0]:.3f}–{TAIL_PEAK_SEARCH_WIN[1]:.3f} s")
    if BASELINE_ALERT:
        print("  WARNING: " + BASELINE_ALERT)

# Visual check plot: the command and the FILTERED current of the last sweep with the windows shaded.
# If SHOW_UNFILTERED = True (section 4c) the original trace is also drawn in light grey.
def verify_windows(path_):
    """Command and current (unfiltered and filtered) of the last sweep, with the windows shaded."""
    abf = pyabf.ABF(path_)
    fs = abf.dataRate
    apply_windows(abf)
    print(f"{os.path.basename(path_)}: {abf.sweepCount} sweeps · {abf.sweepLengthSec:.3f} s per sweep · {fs} Hz")
    print(f"Window mode: {WINDOW_MODE}")
    _window_summary()
    abf.setSweep(abf.sweepCount - 1, channel=CHANNEL)
    t, i, cmd = abf.sweepX, abf.sweepY, abf.sweepC
    y = apply_filter(i, fs)
    fig, ax = plt.subplots(3, 1, figsize=(12, 10))
    ax[1].sharex(ax[0])
    ax[0].plot(t, cmd, color="#000000")
    ax[0].set_ylabel("Command (mV)")
    if SHOW_UNFILTERED:      # the original trace (light grey) is only drawn if requested in section 4c
        ax[1].plot(t, i, color="#cfcfcf", label="Unfiltered")
    ax[1].plot(t, y, color="#1F5FBF", lw=1, label="Filtered" if FILTER_TYPE != "none" else "(no filter)")
    ax[1].set(ylabel="Current (pA)", xlabel="Time (s)")
    zones = [(BASELINE_WIN, "#7f7f7f", "Baseline"), (STEADY_WIN, "#1F5FBF", "Steady state"),
             (TAIL_PEAK_SEARCH_WIN, "#D62728", "Tail-peak search")]
    for (a, b), color, label_text in zones:
        ax[0].axvspan(a, b, color=color, alpha=0.3)
        ax[1].axvspan(a, b, color=color, alpha=0.3, label=label_text)
        ax[2].axvspan(a, b, color=color, alpha=0.3)
    ax[1].legend(loc="upper right", fontsize=8)
    ax[0].set_xlim(0, min(t[-1], TAIL_WIN[1] + 0.5))
    lo, hi = np.percentile(i, [0.5, 99.5])
    margin = 0.15 * (hi - lo)
    ax[1].set_ylim(lo - margin, hi + margin)      # the capacitive transients fall outside the scale
    ax[1].set_title("Current (scaled to the data: capacitive transients fall outside the plot)", fontsize=9)
    z = window_mask(t, TAIL_WIN[0] - 0.03, TAIL_WIN[0] + 0.15)
    if SHOW_UNFILTERED:
        ax[2].plot(t[z], i[z], color="#cfcfcf")
    ax[2].plot(t[z], y[z], color="#1F5FBF")
    ax[2].set(title="Zoom: start of the tail (capacitive transient and peak)", xlabel="Time (s)", ylabel="Current (pA)")
    ax[2].set_xlim(TAIL_WIN[0] - 0.03, TAIL_WIN[0] + 0.15)
    fig.suptitle("Windows and filter check (last sweep)", fontweight="bold")
    fig.tight_layout()
    plt.show()

# Shows the signal before and after subtracting the baseline, and the baseline value in each sweep (it should be stable and close to 0)
def baseline_diagnostics(path_):
    """Top: filtered signal before subtracting the baseline. Middle: after subtracting it.
    Bottom: baseline estimated in each sweep (corrects one offset per sweep, not drift within a sweep)."""
    abf = pyabf.ABF(path_)
    fs = abf.dataRate
    apply_windows(abf)
    print(f"Baseline: window '{BASELINE_USED}' ({BASELINE_WIN[0]:.3f}–{BASELINE_WIN[1]:.3f} s), command at {V_BASELINE:.0f} mV")
    if BASELINE_ALERT:
        print("WARNING: " + BASELINE_ALERT)
    fig, ax = plt.subplots(3, 1, figsize=(12, 11))
    baselines = []
    colors = plt.matplotlib.colors.LinearSegmentedColormap.from_list("nbr", ["#000000", "#1F5FBF", "#D62728"])(np.linspace(0, 1, abf.sweepCount))
    xmax = min(abf.sweepLengthSec, TAIL_WIN[1] + 0.3)
    for s in abf.sweepList:
        abf.setSweep(s, channel=CHANNEL)
        t, i = abf.sweepX, abf.sweepY
        y = apply_filter(i, fs)
        baseline = np.median(i[window_mask(t, *BASELINE_WIN)])
        baselines.append(baseline)
        m = t <= xmax
        ax[0].plot(t[m], y[m], lw=0.6, color=colors[s])
        ax[1].plot(t[m], (y - baseline)[m], lw=0.6, color=colors[s])
    for a in ax[:2]:
        a.axhline(0, color="k", lw=0.7, ls="--")
        a.axvspan(BASELINE_WIN[0], BASELINE_WIN[1], color="#7f7f7f", alpha=0.3)
        a.set_ylabel("Current (pA)")
        a.grid(alpha=0.25)
    ax[0].set_title("Filtered, before baseline subtraction")
    ax[1].set_title("Filtered, baseline subtracted (per sweep)")
    ax[1].set_xlabel("Time (s)")
    ax[2].plot(np.arange(len(baselines)), baselines, "o-", color="#000000")
    ax[2].axhline(0, color="k", lw=0.7, ls="--")
    ax[2].set(title=f"Baseline estimated in each sweep (at {V_BASELINE:.0f} mV)", xlabel="Sweep", ylabel="Current (pA)")
    ax[2].grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

# One row per file with the detected protocol. If a file fails it does not stop: it is left with "ERROR" in the status column.
def protocol_summary(paths_):
    """One row per file: sweeps, duration, detected windows and warnings. It does not stop if a file fails."""
    rows_ = []
    for r in paths_:
        row_ = {"file": os.path.basename(r)}
        try:
            abf = pyabf.ABF(r)
            row_.update(sweeps=abf.sweepCount, duration_s=round(abf.sweepLengthSec, 3), fs_Hz=int(abf.dataRate))
            apply_windows(abf)
            row_.update(baseline_segment=BASELINE_USED, V_baseline_mV=round(V_BASELINE), pulse_s=f"{PULSE_WIN[0]:.3f}–{PULSE_WIN[1]:.3f}",
                        tail_s=f"{TAIL_WIN[0]:.3f}–{TAIL_WIN[1]:.3f}", V_tail_mV=round(V_TAIL),
                        status=("WARNING: " + BASELINE_ALERT) if BASELINE_ALERT else "OK")
        except Exception as e:
            row_["status"] = f"ERROR: {e}"
        rows_.append(row_)
    return pd.DataFrame(rows_)

### 4b. Statistics

In [ ]:
#@title 4b. Statistics functions { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: defines the statistical tests, the effect size and the descriptive tables.
# Rationale: with few cells per group (small n) caution is required: tests that do not assume equal variances (Welch) are used,
# together with a non-parametric alternative (Mann–Whitney), the effect size (Hedges' g, which the p value does not provide) and Holm's correction when several comparisons are made.
# The STATISTICAL UNIT IS THE CELL: sweeps from the same cell are not independent replicates.
# ----------------------------------------------------------------------------------------------------------
# Shapiro–Wilk normality test. With n < 5 it almost never detects anything: interpret it with caution.
def shapiro_p(x):
    x = np.asarray(x, float)
    if len(x) < 3 or np.ptp(x) == 0:
        return np.nan
    try:
        return stats.shapiro(x).pvalue
    except Exception:
        return np.nan

# Holm–Bonferroni correction: adjusts the p values when several comparisons are made, to control the probability of false positives.
def holm(p):
    """Holm–Bonferroni correction (controls the familywise type I error)."""
    p = np.asarray(p, float)
    aj = np.full(len(p), np.nan)
    ok = np.where(np.isfinite(p))[0]
    order_ = ok[np.argsort(p[ok])]
    prev = 0.0
    for span_range, idx in enumerate(order_):
        prev = max(prev, (len(order_) - span_range) * p[idx])
        aj[idx] = min(1.0, prev)
    return aj

# Effect size in units of standard deviation (corrected for small n): ~0.2 small, ~0.5 medium, ~0.8 large. The p value does NOT convey this.
def hedges_g(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    if sp == 0:
        return np.nan
    return (np.mean(a) - np.mean(b)) / sp * (1 - 3 / (4 * (na + nb) - 9))

# Difference of means with its 95 % confidence interval (Welch). If the CI does not contain 0, it agrees with p < 0.05.
def welch_difference(a, b):
    """Difference of means (a − b) with the Welch 95% CI."""
    va, vb = np.var(a, ddof=1) / len(a), np.var(b, ddof=1) / len(b)
    se = np.sqrt(va + vb)
    if se == 0:
        return np.mean(a) - np.mean(b), np.nan, np.nan
    gl = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    tc = stats.t.ppf(0.975, gl)
    d = np.mean(a) - np.mean(b)
    return d, d - tc * se, d + tc * se

# Table per group and variable: n, mean, SD (dispersion between cells), SEM (uncertainty of the mean), CI95, median, quartiles, range and Shapiro.
def descriptive_stats(df, variables):
    rows_ = []
    for g, sub in df.groupby("group"):
        for v in variables:
            x = sub[v].dropna().astype(float).values
            n = len(x)
            if n == 0:
                continue
            sd = np.std(x, ddof=1) if n > 1 else np.nan
            sem = sd / np.sqrt(n) if n > 1 else np.nan
            tc = stats.t.ppf(0.975, n - 1) if n > 1 else np.nan
            rows_.append({"group": g, "variable": v, "n": n, "mean": np.mean(x), "SD": sd, "SEM": sem,
                          "CI95_low": np.mean(x) - tc * sem if n > 1 else np.nan,
                          "CI95_high": np.mean(x) + tc * sem if n > 1 else np.nan,
                          "median": np.median(x), "Q1": np.percentile(x, 25), "Q3": np.percentile(x, 75),
                          "min": np.min(x), "max": np.max(x), "Shapiro_p": shapiro_p(x)})
    return pd.DataFrame(rows_)

# Compares the groups pairwise with Welch and Mann–Whitney, and corrects with Holm. With ≥ 3 groups it adds global ANOVA and Kruskal–Wallis.
def compare_groups(df, variables):
    """Compares groups pairwise (Welch and Mann–Whitney) with Holm correction per variable.
    With ≥3 groups it adds global ANOVA and Kruskal–Wallis."""
    pairs, global_rows = [], []
    for v in variables:
        data = {g: sub[v].dropna().astype(float).values for g, sub in df.groupby("group")}
        data = {g: x for g, x in data.items() if len(x) >= 2}
        if len(data) < 2:
            continue
        if len(data) >= 3:
            try:   p_a = stats.f_oneway(*data.values()).pvalue
            except Exception: p_a = np.nan
            try:   p_k = stats.kruskal(*data.values()).pvalue
            except Exception: p_k = np.nan
            global_rows.append({"variable": v, "n_groups": len(data), "ANOVA_p": p_a, "Kruskal_p": p_k})
        rows_ = []
        for ga, gb in itertools.combinations(data, 2):
            a, b = data[ga], data[gb]
            d, lo, hi = welch_difference(a, b)
            try:   p_w = stats.ttest_ind(a, b, equal_var=False).pvalue
            except Exception: p_w = np.nan
            try:   p_m = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
            except Exception: p_m = np.nan
            try:   p_l = stats.levene(a, b).pvalue
            except Exception: p_l = np.nan
            # If both groups are compatible with normality Welch is suggested; otherwise Mann–Whitney
            normal = all(shapiro_p(x) > 0.05 for x in (a, b))
            rows_.append({"variable": v, "comparison": f"{ga} vs {gb}", "n_a": len(a), "n_b": len(b),
                          "mean_a": np.mean(a), "mean_b": np.mean(b),
                          "mean_diff": d, "CI95_low": lo, "CI95_high": hi, "Hedges_g": hedges_g(a, b),
                          "Levene_p": p_l, "normality_ok": normal,
                          "p_Welch": p_w, "p_MannWhitney": p_m})
        pw = holm([f["p_Welch"] for f in rows_])
        pm = holm([f["p_MannWhitney"] for f in rows_])
        for f, a_, b_ in zip(rows_, pw, pm):
            f["p_Welch_Holm"], f["p_MW_Holm"] = a_, b_
            f["suggested_test"] = "Welch" if f["normality_ok"] else "Mann–Whitney"
            f["p_suggested_Holm"] = a_ if f["normality_ok"] else b_
        pairs.extend(rows_)
    return pd.DataFrame(pairs), pd.DataFrame(global_rows)

# Repeats the comparison at EACH voltage (with Holm across voltages). It is very conservative with small n.
def compare_by_voltage(df_all, variable):
    """At each voltage compares groups (Welch and Mann–Whitney) with Holm across voltages."""
    groups = order_groups(df_all["group"].unique())
    rows_ = []
    for ga, gb in itertools.combinations(groups, 2):
        sub = []
        for v, sv in df_all.groupby("V_rounded"):
            a = sv.loc[sv["group"] == ga, variable].dropna().values
            b = sv.loc[sv["group"] == gb, variable].dropna().values
            if len(a) < 2 or len(b) < 2:
                continue
            try:   p_w = stats.ttest_ind(a, b, equal_var=False).pvalue
            except Exception: p_w = np.nan
            try:   p_m = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
            except Exception: p_m = np.nan
            sub.append({"variable": variable, "comparison": f"{ga} vs {gb}", "V_mV": v,
                        "n_a": len(a), "n_b": len(b), "mean_a": np.mean(a), "mean_b": np.mean(b),
                        "p_Welch": p_w, "p_MannWhitney": p_m})
        if sub:
            pw, pm = holm([s["p_Welch"] for s in sub]), holm([s["p_MannWhitney"] for s in sub])
            for s, a_, b_ in zip(sub, pw, pm):
                s["p_Welch_Holm"], s["p_MW_Holm"] = a_, b_
            rows_ += sub
    return pd.DataFrame(rows_)

# Spearman correlation (does not assume normality) between parameters, within each group with n ≥ 5. Pairs linked by definition are omitted.
def correlations(df, variables):
    """Spearman between parameters. Computed within each group with n ≥ 5; if no group reaches 5
    but the total does, all cells are pooled (and a correlation may reflect the difference between groups, not a real relationship).
    Pairs that are linked by definition (k↔z, pA↔pA/pF) are omitted."""
    trivial_pairs = [{"k_mV", "z_apparent"}, {"I_tail_max_pA", "I_tail_max_pApF"}]
    segments = [(g, sub) for g, sub in df.groupby("group") if len(sub) >= 5]
    if not segments and len(df) >= 5:
        segments = [("all (pooled)", df)]
    rows_ = []
    for g, sub_g in segments:
        for a, b in itertools.combinations(variables, 2):
            if {a, b} in trivial_pairs:
                continue
            sub = sub_g[[a, b]].dropna()
            if len(sub) < 5 or sub[a].nunique() < 2 or sub[b].nunique() < 2:
                continue
            rho, p = stats.spearmanr(sub[a], sub[b])
            rows_.append({"group": g, "variable_1": a, "variable_2": b, "n": len(sub), "rho_Spearman": rho, "p": p})
    return pd.DataFrame(rows_).sort_values("p") if rows_ else pd.DataFrame()

# Detects outliers with the modified z-score (median and MAD, robust): they are FLAGGED for review, never deleted automatically.
def outliers(df, variables, threshold=3.5):
    """Outlying points by modified z-score (based on the median and MAD) within each group."""
    rows_ = []
    for g, sub in df.groupby("group"):
        for v in variables:
            x = sub[["file", v]].dropna()
            if len(x) < 4:
                continue
            med = np.median(x[v])
            mad = np.median(np.abs(x[v] - med))
            if mad == 0:
                continue
            z = 0.6745 * (x[v] - med) / mad
            for abf_name, val, zz in zip(x["file"], x[v], z):
                if abs(zz) > threshold:
                    rows_.append({"group": g, "file": abf_name, "variable": v, "value": val, "z_modified": zz})
    return pd.DataFrame(rows_)

# One row per cell with everything that is compared between groups. Kinetic parameters are taken at the voltage V_REF.
def summary_row(name, res, group):
    df, b, qc = res["df"], res["boltz"], res["qc"]
    r = df.loc[(df["V_test_mV"] - V_REF_mV).abs().idxmin()]
    return {"file": name, "group": group, "n_sweeps": len(df),
            "Cm_used_pF": res["cm_used"], "Cm_estimated_pF": res["cm_est"],
            "V_ref_mV": r["V_test_mV"],
            "I_tail_max_pA": df["I_tail_pA"].max(),
            "I_tail_max_pApF": df["I_tail_pApF"].max() if "I_tail_pApF" in df else np.nan,
            "I_steady_max_pA": df["I_steady_pA"].max(),
            "V50_mV": b["V50_mV"] if b else np.nan, "k_mV": b["k_mV"] if b else np.nan,
            "z_apparent": b["z_apparent"] if b else np.nan, "R2_boltzmann": b["R2"] if b else np.nan,
            "tau_mono_Vref_ms": r["tau_mono_ms"], "tau_fast_Vref_ms": r["tau_fast_ms"],
            "tau_slow_Vref_ms": r["tau_slow_ms"], "t_tail_peak_Vref_ms": r["t_tail_peak_ms"],
            "I_hold_mean_pA": qc["I_hold_mean_pA"], "noise_pA": qc["noise_mean_pA"],
            "drift_pA_per_sweep": qc["drift_pA_per_sweep"], "QC": qc["status"]}

# ---------- Significance between TWO groups (for the figures) ----------
# Converts p into marks: * p<0.05 · ** p<0.01 · *** p<0.001 · ns = not significant (which does NOT mean "equal")
def stars(p):
    """* p<0.05 · ** p<0.01 · *** p<0.001 · ns = not significant."""
    if p is None or not np.isfinite(p):
        return "n/a"
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

def fmt_p(p):
    if p is None or not np.isfinite(p):
        return "p = n/a"
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

# p value of comparing TWO groups with one row per cell. It is a single comparison, so no Holm correction is applied.
def p_two_groups(table, variable, ga, gb, test_name="suggested"):
    """p value of comparing ga vs gb with ONE row per cell. It is a single comparison: no Holm correction.
    test_name = 'suggested' (Welch if both groups are compatible with normality; otherwise Mann–Whitney), 'Welch' or 'Mann-Whitney'."""
    a = table.loc[table["group"] == ga, variable].dropna().astype(float).values
    b = table.loc[table["group"] == gb, variable].dropna().astype(float).values
    out = {"variable": variable, "group_a": ga, "group_b": gb, "n_a": len(a), "n_b": len(b),
           "p": np.nan, "test_name": "", "p_Welch": np.nan, "p_MW": np.nan, "normality_ok": False}
    if len(a) < 2 or len(b) < 2:
        return out
    try:   out["p_Welch"] = stats.ttest_ind(a, b, equal_var=False).pvalue
    except Exception: pass
    try:   out["p_MW"] = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    except Exception: pass
    out["normality_ok"] = bool(all(shapiro_p(x) > 0.05 for x in (a, b)))
    pr = str(test_name).lower()
    uses_welch = False if pr.startswith("mann") else True if pr == "welch" else out["normality_ok"]
    out["test_name"] = "Welch" if uses_welch else "Mann–Whitney"
    out["p"] = out["p_Welch"] if uses_welch else out["p_MW"]
    return out

# Prints the comparison in plain language: means ± SEM, difference with CI95 %, effect size, test used and p.
def summary_two_groups(table, variable, ga, gb, test_name="suggested", unit=""):
    """Summarises in plain language the comparison (gb − ga) of one variable (one row = one cell)."""
    import math
    a = table.loc[table["group"] == ga, variable].dropna().astype(float).values
    b = table.loc[table["group"] == gb, variable].dropna().astype(float).values
    r = p_two_groups(table, variable, ga, gb, test_name)
    print(f"Variable: {variable}")
    for g, x in ((ga, a), (gb, b)):
        sem = np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
        print(f"  {g:<10} n = {len(x)} · mean ± SEM = {np.mean(x):.1f} ± {sem:.1f} {unit}")
    if len(a) < 2 or len(b) < 2:
        print("  At least 2 cells per group are needed to compare.")
        return r
    d, lo, hi = welch_difference(b, a)
    g = hedges_g(b, a)
    ag = abs(g) if np.isfinite(g) else np.nan
    size_ = "large" if ag >= 0.8 else "medium" if ag >= 0.5 else "small" if ag >= 0.2 else "trivial"
    print(f"  Difference ({gb} − {ga}) = {d:+.1f} {unit} · 95 % CI (Welch) [{lo:+.1f}, {hi:+.1f}] · Hedges' g = {g:.2f} ({size_})")
    print(f"  Test used: {r['test_name']} → {fmt_p(r['p'])}  {stars(r['p'])}"
          f"   (Welch p = {r['p_Welch']:.4f} · Mann–Whitney p = {r['p_MW']:.4f} · "
          f"normality compatible in both groups: {'yes' if r['normality_ok'] else 'no'})")
    # With small n, Mann–Whitney has a minimum attainable p (e.g. 3 vs 8 → 0.012): if that minimum is already > 0.05, it can never be significant
    p_min = 2 / math.comb(len(a) + len(b), len(a))
    print(f"  With n = {len(a)} vs {len(b)}, the smallest p that Mann–Whitney can give is {p_min:.4f}.")
    if np.isfinite(r["p"]):
        if r["p"] < 0.05:
            print("  Reading: statistically significant difference (α = 0.05). The 95 % CI and g indicate how large and how precise it is.")
        else:
            print("  Reading: no difference can be claimed with these data (this does not mean they are equal: with small n the power is low).")
    return r

# ---------- Can pA/pF be used? Only if ALL the cells compared have capacitance ----------
# Memory of the warnings already shown (so they are not repeated in every figure)
_CM_WARNINGS_SHOWN = set()
# pA/pF is used only if ALL the cells compared have capacitance; if any is missing a warning is issued and pA is used, so that units are not mixed.
def cm_complete(df, col):
    """True if every cell in df has `col` (the pA/pF column). If any is missing, warns once and returns False
    so that groups with different units are not compared and cells are not silently left out."""
    if col not in df or not df[col].notna().any():
        return False
    missing = sorted(set(df["file"]) - set(df.loc[df[col].notna(), "file"]))
    if missing:
        key_name = (col, tuple(missing))
        if key_name not in _CM_WARNINGS_SHOWN:
            _CM_WARNINGS_SHOWN.add(key_name)
            print(f"WARNING: the capacitance is missing for: {', '.join(missing)}. pA is used (units are not mixed). To express in pA/pF they must be entered in CAPACITANCES_pF.")
        return False
    return True

### 4c. Plots

In [ ]:
#@title 4c. Figures: style, voltage protocol and plots { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: defines the common style and all the functions that draw the figures.
# Rationale: a single palette (black / blue / red), typography and fixed sizes make all the figures consistent and ready for the article.
# Figures are saved as PNG (300 dpi) and vector PDF (editable in Illustrator/Inkscape).
# ----------------------------------------------------------------------------------------------------------
import textwrap
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import binary_dilation

#@markdown **Language of ALL figures**
LANGUAGE = "en"   #@param ["en", "es"]

#@markdown **Main figure (Figure 1), panel A.** Time window shown (s). If the end is ≤ the start, the whole sweep is shown.
FIG_T_START_S = 0.4   #@param {type:"number"}
FIG_T_END_S = 3.0      #@param {type:"number"}
#@markdown Calibration bars of panel A (0 = automatic round values)
SCALE_PA = 0          #@param {type:"number"}
SCALE_MS = 0          #@param {type:"number"}
#@markdown Hide the capacitive artefact at each voltage step in the PLOTS only (the analysis is not modified)
BLANK_TRANSIENTS = True  #@param {type:"boolean"}
#@markdown Write the recording name inside the figures (False is recommended for publication; the name is always printed above each figure)
NAME_IN_FIGURES = False  #@param {type:"boolean"}

#@markdown **Figure 2 (Control vs transfected).** Test used for the p value shown in the figure
SIGNIFICANCE_TEST = "suggested"  #@param ["suggested", "Welch", "Mann-Whitney"]
#@markdown Also mark each voltage on the tail-amplitude curve (Holm correction across voltages; conservative: with small n all usually come out "ns")
SIGNIFICANCE_PER_VOLTAGE = False  #@param {type:"boolean"}

#@markdown **Original curves.** By default the figures show ONLY the mean curves and the filtered traces (no light-grey originals).
#@markdown Show the thin line of each individual cell behind the mean (Figure 2 and group summaries)
SHOW_INDIVIDUAL_CURVES = False  #@param {type:"boolean"}
#@markdown Show the UNFILTERED trace (light grey) behind the filtered one in the window-check plots (section 5)
SHOW_UNFILTERED = False  #@param {type:"boolean"}

# ------------------------------------------------------------------ texts
# Figure texts in English (en) and Spanish (es): LANGUAGE selects which one is used in ALL figures
_TXT = {
    "en": dict(time_s="Time (s)", current="Current (pA)", v_mem="Test potential (mV)", v_cmd="Command (mV)",
               i_norm="Normalized tail current", tail="Tail currents", iv="I–V relationship",
               g="Steady-state conductance", act="Activation curve", deact="Deactivation",
               tau="Deactivation kinetics", baseline="Baseline stability", ss="Steady state",
               peak="Peak during pulse", ptail="Tail peak", data="Data", sweep="Sweep",
               baseline_y="Baseline before subtraction (pA)", tau_y="τ (ms)", g_y="Conductance (nS)",
               ttail="Time from step (ms)", t_ms="Time (ms)", summary="Summary", no_signal="no signal",
               step="ΔV = +{dv:g} mV", n_sweeps="{n} sweeps", group="Group",
               params="Boltzmann fit parameters (mean ± SEM)", file="file", imax="Max tail current",
               filt="filter", mono="τ mono", fast="τ fast", slow="τ slow",
               amp_pf="Tail current (pA/pF)", amp_pa="Tail current (pA)", amp_max_pf="Max tail amplitude (pA/pF)",
               amp_max_pa="Max tail amplitude (pA)", v50="V½ of activation (mV)", k="Slope k (mV)",
               tau_title="Deactivation τ (mono-exp)", amp_vs_v="Tail amplitude vs. voltage",
               act_mean="Activation (mean ± SEM)", all_cells="All cells", filtered="QC-passed cells (controls kept)",
               cat_best="Best {n}", cat_mid="Middle {n}", cat_worst="Worst {n}",
               test_note="{test}, two-sided · * p < 0.05, ** p < 0.01, *** p < 0.001, ns = not significant · each point = one cell (mean ± SEM)",
               step_pa="Step current (pA)", step_pf="Step current (pA/pF)", step_norm="Normalized step current"),
    "es": dict(time_s="Tiempo (s)", current="Corriente (pA)", v_mem="Potencial de la prueba (mV)", v_cmd="Comando (mV)",
               i_norm="Corriente de cola normalizada", tail="Corrientes de cola", iv="Relación I–V",
               g="Conductancia en estado estable", act="Curva de activación", deact="Desactivación",
               tau="Cinética de desactivación", baseline="Estabilidad de la línea base", ss="Estado estable",
               peak="Pico en el pulso", ptail="Pico de cola", data="Datos", sweep="Sweep",
               baseline_y="Línea base antes de restar (pA)", tau_y="τ (ms)", g_y="Conductancia (nS)",
               ttail="Tiempo desde el salto (ms)", t_ms="Tiempo (ms)", summary="Resumen", no_signal="sin señal",
               step="ΔV = +{dv:g} mV", n_sweeps="{n} sweeps", group="Grupo",
               params="Parámetros del ajuste de Boltzmann (media ± EEM)", file="archivo", imax="Cola máx",
               filt="filtro", mono="τ mono", fast="τ rápida", slow="τ lenta",
               amp_pf="Corriente de cola (pA/pF)", amp_pa="Corriente de cola (pA)", amp_max_pf="Amplitud máx. de la cola (pA/pF)",
               amp_max_pa="Amplitud máx. de la cola (pA)", v50="V½ de activación (mV)", k="Pendiente k (mV)",
               tau_title="τ de desactivación (mono-exp)", amp_vs_v="Amplitud de cola vs. voltaje",
               act_mean="Activación (media ± EEM)", all_cells="Todas las células", filtered="Células que pasan el QC (controles incluidos)",
               cat_best="{n} Mejores", cat_mid="{n} Medios", cat_worst="{n} Peores",
               test_note="{test}, bilateral · * p < 0.05, ** p < 0.01, *** p < 0.001, ns = no significativo · cada punto = una célula (media ± EEM)",
               step_pa="Corriente del paso (pA)", step_pf="Corriente del paso (pA/pF)", step_norm="Corriente del paso normalizada"),
}
def T(key_name):
    return _TXT[LANGUAGE][key_name]

# ------------------------------------------------------------------ ONE palette for every figure: black, blue, red
# Single palette: black, blue, red and their light tones. Colour is not used as the only code: markers and line styles also change.
BLACK, BLUE, RED = "#000000", "#1F5FBF", "#D62728"
GREY = "#8c8c8c"
LIGHT_GREY = "#cfcfcf"
PALETTE = [BLACK, BLUE, RED, "#7F7F7F", "#7FA6E0", "#E88D8D"]      # groups: black, blue, red and their light tones
CATEGORY_COLOR = {"best": BLACK, "mid": BLUE, "worst": RED}
CATEGORY_STYLE = {"best": "-", "mid": "--", "worst": ":"}
CATEGORY_MARKER = {"best": "o", "mid": "s", "worst": "^"}
CMAP_SWEEPS = LinearSegmentedColormap.from_list("black_blue_red", [BLACK, BLUE, RED])   # low → high voltage
INK = BLACK

def category_label(key_name, n):
    return T("cat_" + key_name).format(n=n) if key_name in CATEGORY_COLOR else str(key_name)

# Global matplotlib settings: Arial/Helvetica font, axes without top/right box, editable text in PDFs (fonttype 42).
def figure_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8, "axes.labelsize": 8.5, "axes.titlesize": 9, "xtick.labelsize": 7.5,
        "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
        "axes.linewidth": 0.8, "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "xtick.major.size": 3, "ytick.major.size": 3,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
        "text.color": INK, "axes.grid": False, "legend.frameon": False,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
        "pdf.fonttype": 42, "svg.fonttype": "none", "mathtext.default": "regular",
        "axes.unicode_minus": True,
    })
figure_style()

# Labels the panels A, B, C... as in a paper
def panel_letter(ax, letter, dx=-0.14, dy=1.03):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom", ha="left")

# Saves the figure as PNG (300 dpi, for the text) and as vector PDF (for editing).
def save_figure(fig, folder, name):
    os.makedirs(folder, exist_ok=True)
    fig.savefig(os.path.join(folder, name + ".png"), dpi=300)
    fig.savefig(os.path.join(folder, name + ".pdf"))      # vector, editable in Illustrator/Inkscape

def _fmt_dur(ms):
    return f"{ms / 1000:g} s" if ms >= 1000 else f"{ms:g} ms"

def _fmt_mv(x):
    return f"{x:+.0f}".replace("-", "−") + " mV"

def _fmt_num(x, d=1):
    return f"{x:.{d}f}".replace("-", "−")

# Rounds to a "nice" value (1, 2, 2.5, 5 × 10ⁿ) for the calibration bars
def _nice_number(x):
    """The 'round' value (1, 2, 2.5, 5 × 10^n) closest to x, for the calibration bars."""
    e = 10 ** np.floor(np.log10(x))
    return min((m * e for m in (1, 2, 2.5, 5, 10)), key=lambda v: abs(v - x))

# ------------------------------------------------------------------ protocol + traces
# Returns, sorted by voltage: the REAL command of each sweep and the FILTERED current with that sweep's baseline already subtracted (the same one used by the analysis)
def _protocol_data(abf):
    """REAL command of each sweep and filtered current WITH THAT SWEEP'S BASELINE SUBTRACTED
    (the same median of the unfiltered signal that analyze_abf uses). Sorted from lowest to highest voltage.
    Requires apply_windows(abf) to have been called beforehand."""
    cmds, vp, ys = [], [], []
    for s in abf.sweepList:
        abf.setSweep(s, channel=CHANNEL)
        t = np.array(abf.sweepX)
        i_raw = raw_current(abf)
        baseline = np.median(i_raw[window_mask(t, *BASELINE_WIN)])
        cmds.append(np.array(abf.sweepC))
        vp.append(float(np.mean(abf.sweepC[window_mask(t, PULSE_WIN[0] + 0.1, PULSE_WIN[1] - 0.1)])))
        ys.append(apply_filter(i_raw, abf.dataRate) - baseline)
    order_ = np.argsort(vp)
    return t, np.array(cmds)[order_], np.array(vp)[order_], np.array(ys)[order_]

# Defines the segments A (before the pulse), B (pulse), C (tail) and D (end) that are labelled in the protocol
def _epochs(abf):
    L = abf.sweepLengthSec
    a0 = max(0.0, PULSE_WIN[0] - EPOCH_A_MS / 1000)
    return [("A", a0, PULSE_WIN[0]), ("B", PULSE_WIN[0], PULSE_WIN[1]), ("C", TAIL_WIN[0], TAIL_WIN[1]), ("D", TAIL_WIN[1], L)]

# Draws the voltage protocol: black where all the sweeps are identical and coloured where they change (the test pulse)
def draw_protocol(ax, t, cmds, vp, colors, epochs, x0, x1, bars=False):
    span = x1 - x0
    vis = (t >= x0 - 0.005) & (t <= x1 + 0.005)
    # Part of the command equal in all sweeps: black. Part that changes: colour of each sweep.
    varies = np.ptp(cmds, axis=0) > 0.5
    varies_ext = binary_dilation(varies, iterations=1)       # includes the edge so that the vertical jump is visible
    ax.plot(t[vis], np.where(~varies, cmds[0], np.nan)[vis], color=BLACK, lw=1.4)
    for c, col in zip(cmds, colors):
        ax.plot(t[vis], np.where(varies_ext, c, np.nan)[vis], color=col, lw=1.1)
    ymin, ymax = float(cmds.min()), float(cmds.max())
    ax.set_xlim(x0, x1)
    ax.set_ylim(ymin - 26, ymax + 66)

    # epoch brackets: letter (+ duration if the epoch is fully visible)
    y_br = ymax + 32
    for letter, a, b in epochs:
        ia, ib = max(a, x0), min(b, x1)
        if ib <= ia:
            continue
        cl, cr = a < x0, b > x1                      # epoch clipped by the window
        p = 0.008 * span
        xl, xr = ia + (0 if cl else p), ib - (0 if cr else p)
        pts = ([] if cl else [(xl, y_br - 5)]) + [(xl, y_br), (xr, y_br)] + ([] if cr else [(xr, y_br - 5)])
        ax.plot([q[0] for q in pts], [q[1] for q in pts], color=GREY, lw=0.8, clip_on=False)
        xc = (ia + ib) / 2
        if not cl and not cr:
            ax.text(xc, y_br + 4, _fmt_dur((b - a) * 1000), ha="center", va="bottom", fontsize=6.8, color=GREY)
        ax.text(xc, y_br + 19, letter, ha="center", va="bottom", fontsize=8, fontweight="bold")

    # voltage levels
    v_hold = float(np.median(cmds[0][window_mask(t, x0, x0 + 0.05)]))      # level at the left edge of the drawn part
    v_tail = float(np.median(cmds[0][window_mask(t, TAIL_WIN[0] + 0.05, TAIL_WIN[1] - 0.05)]))
    ax.text(x0 + 0.012 * span, v_hold - 7, _fmt_mv(v_hold), ha="left", va="top", fontsize=7)
    xt = (TAIL_WIN[0] + min(TAIL_WIN[1], x1)) / 2
    ax.text(xt, v_tail - 7, _fmt_mv(v_tail), ha="center", va="top", fontsize=7)
    xm = (PULSE_WIN[0] + PULSE_WIN[1]) / 2
    ax.text(xm, vp[-1] + 6, _fmt_mv(vp[-1]), ha="center", va="bottom", fontsize=7, color=colors[-1], fontweight="bold")
    ax.text(xm, vp[0] - 7, _fmt_mv(vp[0]), ha="center", va="top", fontsize=7, color=colors[0], fontweight="bold")
    dv = float(np.median(np.diff(vp))) if len(vp) > 1 else 0
    ax.text(TAIL_WIN[0] + 0.03 * span, (vp[0] + vp[-1]) / 2 + 8,
            T("step").format(dv=dv) + "\n" + T("n_sweeps").format(n=len(vp)),
            ha="left", va="center", fontsize=6.8, color=GREY, linespacing=1.5)

    if bars:                                    # without axes: only a voltage calibration bar
        ax.axis("off")
        xb = x0 - 0.03 * span
        ax.plot([xb, xb], [v_hold, v_hold + 50], color=BLACK, lw=1.6, clip_on=False, solid_capstyle="butt")
        ax.text(xb - 0.012 * span, v_hold + 25, "50 mV", rotation=90, ha="right", va="center", fontsize=7)
    else:
        ax.set_yticks([v for v in (-80, -40, 0, 40) if ymin - 15 <= v <= ymax + 5])
        ax.set_ylabel(T("v_cmd"))
        ax.tick_params(labelbottom=False, axis="x", length=0)
        ax.spines["bottom"].set_visible(False)

# If BLANK_TRANSIENTS = True the capacitive artefacts of each step are hidden ONLY IN THE PLOT (the analysis is untouched)
def _trace_mask(t, x0, x1, jumps):
    """Visible samples (within the window) and samples that are hidden (capacitive artefact around each step)."""
    hidden = np.zeros_like(t, bool)
    if BLANK_TRANSIENTS:
        for tj in jumps:
            hidden |= window_mask(t, tj - 2 * JUMP_MARGIN_MS / 1000, tj + 2 * JUMP_MARGIN_MS / 1000)
    vis = (t >= x0 - 0.005) & (t <= x1 + 0.005)
    return vis, hidden

def _trace_range(t, ys, x0, x1, jumps):
    vis, hidden = _trace_mask(t, x0, x1, jumps)
    ok = vis & ~hidden
    return float(np.nanmin(ys[:, ok])), float(np.nanmax(ys[:, ok]))

# Draws the filtered traces with an L-shaped calibration bar (pA and ms) instead of axes, as in electrophysiology papers
def draw_traces(ax, t, ys, colors, epochs, x0, x1, bars=False, pad=0.06, lim=None, s_pa_fixed=None, jumps=None):
    """lim=(lo, hi) and s_pa_fixed force the SAME vertical scale and the same pA bar in several panels (control vs transfected).
    jumps = times of the voltage steps of THIS cell (by default, those of the global windows)."""
    span = x1 - x0
    if jumps is None:
        jumps = (PULSE_WIN[0], TAIL_WIN[0], TAIL_WIN[1])
    vis, hidden = _trace_mask(t, x0, x1, jumps)
    for y, col in zip(ys, colors):
        ax.plot(t[vis], np.where(hidden, np.nan, y)[vis], color=col, lw=0.8)
    lo, hi = lim if lim is not None else _trace_range(t, ys, x0, x1, jumps)
    rng = hi - lo
    ax.set_xlim(x0, x1)
    ax.axhline(0, color=GREY, lw=0.6, ls=(0, (4, 3)), zorder=0)
    if not bars:
        ax.set_ylim(lo - pad * rng, hi + pad * rng)
        ax.set_xlabel(T("time_s")); ax.set_ylabel(T("current"))
        ax.spines["bottom"].set_position(("outward", 3)); ax.spines["left"].set_position(("outward", 3))
        return
    # --- L-shaped calibration bar (pA upwards, ms sideways), outside the traces ---
    s_pa = SCALE_PA or s_pa_fixed or _nice_number(0.25 * rng)
    s_ms = SCALE_MS if SCALE_MS else _nice_number(0.20 * span * 1000)
    ax.set_ylim(lo - 0.16 * rng, hi + pad * rng)
    xb, yb = x0 - 0.03 * span, lo - 0.11 * rng
    ax.axis("off")
    ax.plot([xb, xb], [yb, yb + s_pa], color=BLACK, lw=1.6, clip_on=False, solid_capstyle="butt")
    ax.plot([xb, xb + s_ms / 1000], [yb, yb], color=BLACK, lw=1.6, clip_on=False, solid_capstyle="butt")
    ax.text(xb - 0.012 * span, yb + s_pa / 2, f"{s_pa:g} pA", rotation=90, ha="right", va="center", fontsize=7.5)
    ax.text(xb + s_ms / 2000, yb - 0.025 * rng, f"{s_ms:g} ms", ha="center", va="top", fontsize=7.5)
    ax.text(x1 + 0.012 * span, 0, "0 pA", ha="left", va="center", fontsize=6.8, color=GREY)

def panel_protocol_traces(fig, subspec, abf, letter="A", x0=None, x1=None, bars=False, heights=(1, 2.7), hspace=0.06):
    """Protocol (top) and traces (bottom) with the SAME time axis."""
    apply_windows(abf)
    t, cmds, vp, ys = _protocol_data(abf)
    L = abf.sweepLengthSec
    x0 = 0.0 if x0 is None else x0
    x1 = L if (x1 is None or x1 <= x0) else min(x1, L)
    colors = CMAP_SWEEPS(np.linspace(0.0, 1.0, len(vp)))
    ep = _epochs(abf)
    gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=subspec, height_ratios=heights, hspace=hspace)
    ax_p = fig.add_subplot(gs[0])
    ax_t = fig.add_subplot(gs[1], sharex=ax_p)
    draw_protocol(ax_p, t, cmds, vp, colors, ep, x0, x1, bars)
    draw_traces(ax_t, t, ys, colors, ep, x0, x1, bars)
    for _, a, b in ep[1:]:                                    # vertical guides at the voltage jumps
        if x0 < a < x1:
            for a_ in (ax_p, ax_t):
                a_.axvline(a, color=LIGHT_GREY, lw=0.6, ls=":", zorder=0)
    panel_letter(ax_p, letter, dx=-0.10, dy=0.98)
    ax_p.set_xlim(x0, x1)
    return ax_p, ax_t, colors

# ------------------------------------------------------------------ panel A with TWO cells (transfected + control)
# Same voltage = same colour in the transfected cell and in the control, so they can be compared at a glance
def _colors_by_voltage(vp, vmin, vmax):
    """Same voltage = same colour in both cells (the colour scale is set by the transfected cell)."""
    z = (np.asarray(vp, float) - vmin) / (vmax - vmin) if vmax > vmin else np.zeros(len(vp))
    return CMAP_SWEEPS(np.clip(z, 0, 1))

def _prepare_cell(path_):
    """Reads the file, computes ITS windows and returns command + traces (with each sweep's baseline subtracted)."""
    abf = pyabf.ABF(path_)
    apply_windows(abf)
    t, cmds, vp, ys = _protocol_data(abf)
    return {"name": os.path.splitext(os.path.basename(path_))[0], "abf": abf, "t": t, "cmds": cmds, "vp": vp, "ys": ys,
            "epochs": _epochs(abf), "jumps": (PULSE_WIN[0], TAIL_WIN[0], TAIL_WIN[1]), "pulse0": PULSE_WIN[0]}

def panel_protocol_traces_two(fig, subspec, ph, pc, x0, x1, letter="A", labels_=None, hspace=0.07):
    """Protocol (top) + traces of the transfected cell + traces of the control, with the SAME pA and ms scale.
    The control is aligned in time with the onset of the pulse of the transfected cell."""
    labels_ = labels_ or (TRANSFECTED_GROUP, CONTROL_GROUP)
    L = ph["abf"].sweepLengthSec
    x0 = 0.0 if x0 is None else x0
    x1 = L if (x1 is None or x1 <= x0) else min(x1, L)
    # The control is aligned in time with the onset of the transfected-cell pulse (their protocols may have different timings)
    time_shift = ph["pulse0"] - pc["pulse0"]
    tc, jumps_c = pc["t"] + time_shift, tuple(sv + time_shift for sv in pc["jumps"])
    vmin, vmax = float(ph["vp"][0]), float(ph["vp"][-1])
    col_h, col_c = _colors_by_voltage(ph["vp"], vmin, vmax), _colors_by_voltage(pc["vp"], vmin, vmax)
    lo_h, hi_h = _trace_range(ph["t"], ph["ys"], x0, x1, ph["jumps"])
    lo_c, hi_c = _trace_range(tc, pc["ys"], x0, x1, jumps_c)
    # SAME vertical scale and same pA bar in both panels: otherwise the amplitude of control and transfected could not be compared
    lim = (min(lo_h, lo_c), max(hi_h, hi_c))                 # same vertical scale in both
    s_pa = SCALE_PA or _nice_number(0.25 * (lim[1] - lim[0]))    # same calibration bar in both
    gs = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=subspec, height_ratios=(1.0, 1.35, 1.35), hspace=hspace)
    ax_p = fig.add_subplot(gs[0])
    ax_h = fig.add_subplot(gs[1], sharex=ax_p)
    ax_c = fig.add_subplot(gs[2], sharex=ax_p)
    apply_windows(ph["abf"])        # draw_protocol reads the global windows: they must be those of the transfected cell
    draw_protocol(ax_p, ph["t"], ph["cmds"], ph["vp"], col_h, ph["epochs"], x0, x1, bars=True)
    draw_traces(ax_h, ph["t"], ph["ys"], col_h, ph["epochs"], x0, x1, bars=True, lim=lim, s_pa_fixed=s_pa, jumps=ph["jumps"])
    draw_traces(ax_c, tc, pc["ys"], col_c, ph["epochs"], x0, x1, bars=True, lim=lim, s_pa_fixed=s_pa, jumps=jumps_c)
    for a_, lab, nm in ((ax_h, labels_[0], ph["name"]), (ax_c, labels_[1], pc["name"])):
        a_.text(0.005, 0.99, lab, transform=a_.transAxes, ha="left", va="top", fontsize=8.5, fontweight="bold")
        if NAME_IN_FIGURES:
            a_.text(0.005, 0.80, nm, transform=a_.transAxes, ha="left", va="top", fontsize=6.3, color=GREY)
    for _, a, b in ph["epochs"][1:]:                          # vertical guides at the voltage steps
        if x0 < a < x1:
            for a_ in (ax_p, ax_h, ax_c):
                a_.axvline(a, color=LIGHT_GREY, lw=0.6, ls=":", zorder=0)
    panel_letter(ax_p, letter, dx=-0.10, dy=0.98)
    ax_p.set_xlim(x0, x1)
    return ax_p, ax_h, ax_c

# ------------------------------------------------------------------ significance marks
# Bracket with stars and p between two groups
def significance_bracket(ax, x1, x2, y, text_, h, fs=8):
    """Bracket between two groups with the text (stars and p) above."""
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color=BLACK, lw=0.9, clip_on=False, solid_capstyle="butt")
    ax.text((x1 + x2) / 2, y + h * 1.15, text_, ha="center", va="bottom", fontsize=fs)

# ------------------------------------------------------------------ points per group
# One point per cell (with a little horizontal jitter so they do not overlap) and the group mean ± SEM. n is shown below each group.
def points_by_group(ax, table, col, ylabel, colors=None, order_=None):
    """One cell = one point; bar = group mean ± SEM. n is shown below each group."""
    order_ = order_ or globals().get("groups") or list(table["group"].unique())
    colors = colors if colors is not None else PALETTE
    rng = np.random.default_rng(0)
    for i, g in enumerate(order_):
        x = table.loc[table["group"] == g, col].dropna().astype(float).values
        c = colors[i % len(colors)]
        if len(x) == 0:
            continue
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(x)), x, s=24, color=c, edgecolor="white", linewidth=0.6, zorder=3)
        sem = np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0
        ax.errorbar(i + 0.28, np.mean(x), yerr=sem, color=BLACK, marker="_", ms=9, mew=1.4, capsize=3, lw=1.2, zorder=4)
    ax.set_xticks(range(len(order_)))
    ax.set_xticklabels([f"{g}\n(n={int(table.loc[table['group'] == g, col].notna().sum())})" for g in order_])
    ax.set_xlim(-0.6, len(order_) - 0.4)
    ax.set_ylabel(ylabel)

# ------------------------------------------------------------------ mean curves per group (sections 10 and 10b)
# Mean curves per group (amplitude, activation with Boltzmann, τ) and V½, k and amplitude per cell. Individual curves appear only if SHOW_INDIVIDUAL_CURVES = True.
def figure_groups(curve_axes, point_axes, df_t, res_t, prefix=""):
    """Row of 3 curves (amplitude, activation, τ) and row of 3 dot plots. Returns the mean curves."""
    group_list = globals().get("groups") or list(res_t["group"].unique())
    pf = cm_complete(df_t, "I_tail_pApF")
    col_amp = "I_tail_pApF" if pf else "I_tail_pA"
    amp_summary = "I_tail_max_pApF" if pf else "I_tail_max_pA"
    curves = []
    for gi, g in enumerate(group_list):
        sub_g = df_t[df_t["group"] == g]
        if sub_g.empty:
            continue
        c = PALETTE[gi % len(PALETTE)]
        n_cells = sub_g["file"].nunique()
        for panel, col in [(0, col_amp), (1, "tail_norm"), (2, "tau_mono_ms")]:
            if SHOW_INDIVIDUAL_CURVES:      # thin lines = each cell separately
                for _, sc in sub_g.groupby("file"):
                    curve_axes[panel].plot(sc["V_rounded"], sc[col], color=c, alpha=0.18, lw=0.8)
            agg = sub_g.groupby("V_rounded")[col].agg(["mean", "sem", "count"])
            curve_axes[panel].errorbar(agg.index, agg["mean"], yerr=agg["sem"], color=c, marker="o", ms=4, lw=1.2,
                                        capsize=2.5, label=f"{g} (n={n_cells})")
            curves.append(agg.reset_index().assign(group=g, variable=col))
            if panel == 1:
                bm = fit_boltzmann(agg.index.values, agg["mean"].values)
                if bm:
                    vv = np.linspace(agg.index.min(), agg.index.max(), 200)
                    curve_axes[1].plot(vv, boltzmann(vv, bm["A"], bm["V50_mV"], bm["k_mV"]), color=c, ls="--", lw=1.1,
                                        label=f"{g}: V½ = {_fmt_num(bm['V50_mV'])} mV, k = {_fmt_num(bm['k_mV'])} mV")
    curve_axes[0].set(title=prefix + T("amp_vs_v"), xlabel=T("v_mem"), ylabel=T("amp_pf") if pf else T("amp_pa"))
    curve_axes[1].set(title=prefix + T("act_mean"), xlabel=T("v_mem"), ylabel=T("i_norm"))
    curve_axes[2].set(title=prefix + T("tau_title"), xlabel=T("v_mem"), ylabel=T("tau_y"), yscale="log")
    for a in curve_axes:
        a.legend(fontsize=6.8)
        a.title.set_fontsize(8.5)
    points_by_group(point_axes[0], res_t, "V50_mV", T("v50"), PALETTE)
    points_by_group(point_axes[1], res_t, "k_mV", T("k"), PALETTE)
    points_by_group(point_axes[2], res_t, amp_summary, T("amp_max_pf") if pf else T("amp_max_pa"), PALETTE)
    for a, tt in zip(point_axes, [T("v50"), T("k"), T("amp_max_pf") if pf else T("amp_max_pa")]):
        a.set_title(prefix + tt.split(" (")[0], fontsize=8.5)
    return pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()

# ------------------------------------------------------------------ figure of ONE cell (section 6)
# 9-panel figure of ONE cell: protocol and traces, tails, I–V, conductance, activation, deactivation, τ, baseline and summary.
def plot_cell(path_, res, folder="results"):
    figure_style()
    name = os.path.splitext(os.path.basename(path_))[0]
    df, b, ej, qc = res["df"], res["boltz"], res["example"], res["qc"]
    abf = pyabf.ABF(path_)

    fig = plt.figure(figsize=(16, 9.2))
    gs = fig.add_gridspec(2, 5, height_ratios=[1.45, 1], wspace=0.42, hspace=0.40, left=0.05, right=0.985, top=0.93, bottom=0.07)
    apply_windows(abf)
    x0_a = max(0.0, PULSE_WIN[0] - 0.15)        # starts just before the pulse, as in the article figure
    ax_p, ax_t, colors = panel_protocol_traces(fig, gs[0, 0:2], abf, "A", x0_a, None, bars=True, heights=(1.25, 2.2))
    ax = {k: fig.add_subplot(gs[f]) for k, f in
          dict(tail=(0, 2), iv=(0, 3), g=(0, 4), act=(1, 0), deact=(1, 1), tau=(1, 2), baseline=(1, 3), txt=(1, 4)).items()}

    # B. aligned tails (with each sweep's baseline subtracted)
    vp_order = np.sort(df["V_test_mV"].values)
    for s in abf.sweepList:
        abf.setSweep(s, channel=CHANNEL)
        t_, i_raw = abf.sweepX, raw_current(abf)
        yf = apply_filter(i_raw, abf.dataRate) - np.median(i_raw[window_mask(t_, *BASELINE_WIN)])
        k = int(np.where(vp_order == df.loc[df["sweep"] == s, "V_test_mV"].values[0])[0][0])
        m = window_mask(t_, TAIL_WIN[0] + 2 * JUMP_MARGIN_MS / 1000, TAIL_WIN[1] - 2 * JUMP_MARGIN_MS / 1000)
        ax["tail"].plot((t_[m] - TAIL_WIN[0]) * 1000, yf[m], lw=0.8, color=colors[k])
    ax["tail"].axhline(0, color=GREY, lw=0.6, ls=(0, (4, 3)), zorder=0)
    ax["tail"].set(xlabel=T("ttail"), ylabel=T("current"), title=f"{T('tail')} ({_fmt_mv(V_TAIL)})")

    # C. I–V
    a = ax["iv"]
    a.axhline(0, color=GREY, lw=0.6, ls=(0, (4, 3)))
    a.axvline(EK_mV, color=GREY, lw=1, ls=":", label=f"E$_K$ = {_fmt_num(EK_mV)} mV")
    a.plot(df["V_test_mV"], df["I_steady_pA"], "o-", ms=4, lw=1, color=BLACK, mfc="white", mew=1.1, label=T("ss"))
    a.plot(df["V_test_mV"], df["I_pulse_peak_pA"], "s-", ms=4, lw=1, color=BLUE, mfc="white", mew=1.1, label=T("peak"))
    a.plot(df["V_test_mV"], df["I_tail_pA"], "^-", ms=4.5, lw=1, color=RED, mfc="white", mew=1.1, label=T("ptail"))
    a.set(xlabel=T("v_mem"), ylabel=T("current"), title=T("iv"))
    a.legend(loc="upper left", fontsize=7)

    # D. conductance
    a = ax["g"]
    a.plot(df["V_test_mV"], df["G_steady_nS"], "o-", ms=4, lw=1, color=BLACK, mfc="white", mew=1.1)
    a.set(xlabel=T("v_mem"), ylabel=T("g_y"), title=T("g"))

    # E. activation
    a = ax["act"]
    a.axhline(0.5, color=LIGHT_GREY, lw=0.7, ls=":")
    a.plot(df["V_test_mV"], df["tail_norm"], "o", ms=5, color=BLACK, mfc="white", mew=1.1, label=T("data"), zorder=3)
    if b:
        vv = np.linspace(df["V_test_mV"].min() - 5, df["V_test_mV"].max() + 5, 200)
        a.plot(vv, boltzmann(vv, b["A"], b["V50_mV"], b["k_mV"]), color=RED, lw=1.5,
               label=f"V½ = {_fmt_num(b['V50_mV'])} mV\nk = {_fmt_num(b['k_mV'])} mV")
        a.axvline(b["V50_mV"], color=RED, lw=0.7, ls=":")
    a.set(xlabel=T("v_mem"), ylabel=T("i_norm"), title=T("act"), ylim=(-0.05, 1.12))
    a.legend(loc="upper left", fontsize=7)

    # F. deactivation (largest tail)
    a = ax["deact"]
    if ej:
        f = ej["fit"]
        a.plot(ej["t"], ej["y"], color=BLACK, lw=0.9, label=T("data"))      # fit data in thin black (no longer light grey)
        a.plot(ej["t"], model_monoexp(ej["t"], *f["p_mono"]), color=BLUE, lw=1.3,
               label="mono-exp" + (f" (τ = {f['tau_mono_ms']:.0f} ms)" if np.isfinite(f["tau_mono_ms"]) else ""))
        if "p_bi" in f:
            lab = f"bi-exp (τ$_f$ = {f['tau_fast_ms']:.0f}, τ$_s$ = {f['tau_slow_ms']:.0f} ms)" if f["model"] == "bi-exp" else "bi-exp (not preferred)" if LANGUAGE == "en" else "bi-exp (no preferida)"
            a.plot(ej["t"], model_biexp(ej["t"], *f["p_bi"]), color=RED, ls="--", lw=1.3, label=lab)
        a.legend(loc="upper right", fontsize=7)
        a.set_title(f"{T('deact')} ({_fmt_mv(ej['V'])} → {_fmt_mv(V_TAIL)})")
    else:
        a.text(0.5, 0.5, T("no_signal"), transform=a.transAxes, ha="center", color=GREY)
        a.set_title(T("deact"))
    a.set(xlabel=T("t_ms"), ylabel=T("current"))

    # G. τ vs voltage
    a = ax["tau"]
    exists_ = False
    for col, mk, c, lab in [("tau_mono_ms", "o-", BLACK, T("mono")), ("tau_fast_ms", "^--", RED, T("fast")),
                            ("tau_slow_ms", "s--", BLUE, T("slow"))]:
        if df[col].notna().any():
            a.plot(df["V_test_mV"], df[col], mk, ms=4, lw=0.9, color=c, mfc="white", mew=1.1, label=lab)
            exists_ = True
    if exists_:
        a.set_yscale("log")
        a.legend(loc="best", fontsize=7)
    else:
        a.text(0.5, 0.5, T("no_signal"), transform=a.transAxes, ha="center", color=GREY)
    a.set(xlabel=T("v_mem"), ylabel=T("tau_y"), title=T("tau"))

    # H. baseline (value measured in each sweep BEFORE subtracting it)
    a = ax["baseline"]
    a.errorbar(df["sweep"], df["I_baseline_pA"], yerr=df["noise_filtered_pA"], fmt="o-", ms=4, lw=0.9, color=BLACK,
               ecolor=LIGHT_GREY, elinewidth=0.9, capsize=2, mfc="white", mew=1.1)
    a.set(xlabel=T("sweep"), ylabel=T("baseline_y"), title=T("baseline"))

    # I. text summary
    a = ax["txt"]
    a.axis("off")
    lines_ = [(T("file"), name)] if NAME_IN_FIGURES else []
    if res["cm_used"]:
        lines_.append(("C$_m$", f"{res['cm_used']:.1f} pF"))
    if b:
        lines_ += [("V½", f"{_fmt_num(b['V50_mV'])} ± {_fmt_num(b['SE_V50'])} mV"),
                   ("k", f"{_fmt_num(b['k_mV'])} ± {_fmt_num(b['SE_k'])} mV"),
                   ("z", f"{b['z_apparent']:.2f}"), ("R²", f"{b['R2']:.3f}")]
    lines_.append((T("imax"), f"{df['I_tail_pA'].max():.0f} pA"))
    y = 0.98
    a.text(0, y, T("summary"), fontsize=9, fontweight="bold", va="top", transform=a.transAxes)
    y -= 0.12
    for k, v in lines_:
        a.text(0, y, k, fontsize=8, color=GREY, va="top", transform=a.transAxes)
        a.text(0.50, y, v, fontsize=8, va="top", transform=a.transAxes)
        y -= 0.095

    for k, letter in [("tail", "B"), ("iv", "C"), ("g", "D"), ("act", "E"), ("deact", "F"), ("tau", "G"), ("baseline", "H")]:
        panel_letter(ax[k], letter)
        title_ = ax[k].get_title()
        ax[k].set_title("")
        ax[k].set_title(title_, loc="left", fontsize=8.5)
    fig.suptitle((f"{name} · " if NAME_IN_FIGURES else "") + f"{T('filt').capitalize()}: {str(FILTER_TYPE).capitalize()} {CUTOFF_FREQUENCY_HZ:g} Hz", x=0.05, ha="left", fontsize=10, fontweight="bold")
    save_figure(fig, folder, name)
    plt.show()

# ------------------------------------------------------------------ current vs voltage: steps and tails (as in a paper)
# Current vs voltage (step or tail): mean ± error across cells, only at voltages with ≥ 2 cells.
def panel_iv(ax, df_t, kind, groups, colors, styles=None, marks=None, labels_=None, letter=None,
             dispersion="SEM", individual=False, signif_between=None):
    """Current vs voltage, mean ± error across cells (voltages with ≥ 2 cells are plotted).
    kind = 'step' (current at the end of the pulse) or 'tail' (tail peak) · groups = {key: [files]}.
    signif_between = (key_a, key_b) marks with * the voltages with a significant difference (only if SIGNIFICANCE_PER_VOLTAGE = True)."""
    baseline = {"step": "I_steady", "tail": "I_tail"}[kind]
    pf = cm_complete(df_t[df_t["file"].isin([a for l in groups.values() for a in l])], baseline + "_pApF")
    col = baseline + ("_pApF" if pf else "_pA")
    uses_sd = str(dispersion).upper() in ("SD", "SD")
    ax.axhline(0, color=GREY, lw=0.6, ls=(0, (4, 3)), zorder=0)
    aggs = {}
    for k, items in groups.items():
        sub = df_t[df_t["file"].isin(items)]
        c = colors[k]
        if individual and SHOW_INDIVIDUAL_CURVES:
            for _, sc in sub.groupby("file"):
                ax.plot(sc["V_rounded"], sc[col], color=c, alpha=0.18, lw=0.7, zorder=1)
        agg = sub.groupby("V_rounded")[col].agg(["mean", "std", "sem", "count"])
        agg = agg[agg["count"] >= 2]
        aggs[k] = agg
        err = (agg["std"] if uses_sd else agg["sem"]).fillna(0)
        ax.errorbar(agg.index, agg["mean"], yerr=err, marker=(marks or {}).get(k, "o"), ms=4.2, color=c, mfc="white", mew=1.2,
                    lw=1.3, ls=(styles or {}).get(k, "-"), capsize=2, elinewidth=0.9,
                    label=(labels_ or {}).get(k, str(k)), zorder=3)
    if signif_between and SIGNIFICANCE_PER_VOLTAGE:
        ka, kb = signif_between
        key_map = {a: k for k, l in groups.items() for a in l}
        tmp = df_t[df_t["file"].isin(key_map)].copy()
        tmp["group"] = tmp["file"].map(lambda a: str(key_map[a]))
        cv = compare_by_voltage(tmp[tmp["group"].isin([str(ka), str(kb)])], col)
        if len(cv):
            pcol = "p_MW_Holm" if str(SIGNIFICANCE_TEST).lower().startswith("mann") else "p_Welch_Holm"
            yr = ax.get_ylim()[1] - ax.get_ylim()[0]
            for _, r in cv.iterrows():
                if np.isfinite(r[pcol]) and r[pcol] < 0.05:
                    tops = [a_.loc[r["V_mV"], "mean"] + np.nan_to_num(a_.loc[r["V_mV"], "sem"]) for a_ in aggs.values() if r["V_mV"] in a_.index]
                    if tops:
                        ax.text(r["V_mV"], max(tops) + 0.02 * yr, stars(r[pcol]), ha="center", va="bottom", fontsize=8)
    ax.set(xlabel=T("v_mem"), ylabel=T(("step" if kind == "step" else "amp") + ("_pf" if pf else "_pa")))
    ax.spines["bottom"].set_position(("outward", 3)); ax.spines["left"].set_position(("outward", 3))
    # Both curves (step and tail) are low on the left: the upper-left corner is the free area.
    ax.legend(loc="upper left", handlelength=2.2, borderaxespad=0.3)
    if letter:
        panel_letter(ax, letter, dx=-0.2, dy=1.0)

# ------------------------------------------------------------------ FIGURE 1 (paper): A | B + C on a single canvas
# FIGURE 1 (with table): A = traces, B = activation by quality category, C = Boltzmann table, D–E = step and tail current.
def figure_main(path_, categories=None, table=None, folder="results", name="Figure1_hERG", control_path=None, with_iv=True):
    """Left half (50 %): A = real protocol + traces of the best transfected recording (and, if control_path is given,
       those of the control below with the SAME scale) with the baseline subtracted and calibration bars.
       Right half (50 %): B = activation curves (legend at bottom right, where there are no curves) · C = Boltzmann table."""
    figure_style()
    table = summary if table is None else table
    if categories is None:
        categories = {g: sub["file"].tolist() for g, sub in table.groupby("group")}
    category_colors = {k: CATEGORY_COLOR.get(k, PALETTE[i % len(PALETTE)]) for i, k in enumerate(categories)}
    styles = {k: CATEGORY_STYLE.get(k, "-") for k in categories}
    marks = {k: CATEGORY_MARKER.get(k, "o") for k in categories}
    with_control = control_path is not None

    h_top, h_iv = (5.2 if with_control else 4.2), 2.9
    if with_iv:      # bottom row: step current and tail current vs voltage
        fig = plt.figure(figsize=(7.6, h_top + h_iv + 0.6))
        outer = fig.add_gridspec(2, 1, height_ratios=[h_top, h_iv], hspace=0.14, left=0.075, right=0.985, top=0.97, bottom=0.07)
        gs = outer[0].subgridspec(1, 2, width_ratios=[1, 1], wspace=0.24)
    else:
        fig = plt.figure(figsize=(7.6, h_top))
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.24, left=0.075, right=0.985, top=0.94, bottom=0.11)
    if with_control:
        panel_protocol_traces_two(fig, gs[0], _prepare_cell(path_), _prepare_cell(control_path), FIG_T_START_S, FIG_T_END_S, "A")
    else:
        panel_protocol_traces(fig, gs[0], pyabf.ABF(path_), "A", FIG_T_START_S, FIG_T_END_S, bars=True, heights=(1.0, 2.1), hspace=0.02)

    gd = gs[1].subgridspec(2, 1, height_ratios=[1.45, 1], hspace=0.62)
    # B. activation
    ax = fig.add_subplot(gd[0])
    ax.axhline(0.5, color=LIGHT_GREY, lw=0.7, ls=":", zorder=0)
    rows_ = []
    for k, items in categories.items():
        c = category_colors[k]
        lab = category_label(k, len(items))
        dfc = df_all[df_all["file"].isin(items)]
        agg = dfc.groupby("V_rounded")["tail_norm"].agg(["mean", "sem"]).dropna(subset=["mean"])
        ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"].fillna(0), fmt=marks[k], ms=4.2, color=c, mfc="white", mew=1.2,
                    capsize=2, elinewidth=0.9, label=lab, zorder=3)
        bm = fit_boltzmann(agg.index.values, agg["mean"].values)
        if bm:
            vv = np.linspace(agg.index.min() - 5, agg.index.max() + 5, 200)
            ax.plot(vv, boltzmann(vv, bm["A"], bm["V50_mV"], bm["k_mV"]), color=c, ls=styles[k], lw=1.5)
        sub = table[table["file"].isin(items)]
        def ms(col, d=1):
            x = sub[col].dropna()
            if len(x) == 0:
                return "–"
            return _fmt_num(x.mean(), d) + (f" ± {_fmt_num(x.sem(), d)}" if len(x) > 1 else "")
        rows_.append([lab, str(len(sub)), ms("V50_mV"), ms("k_mV"), _fmt_num(sub["R2_boltzmann"].mean(), 3), c])
    ax.set(xlabel=T("v_mem"), ylabel=T("i_norm"), ylim=(-0.06, 1.15))
    ax.spines["bottom"].set_position(("outward", 3)); ax.spines["left"].set_position(("outward", 3))
    # The activation curve rises from left to right: the lower-right corner is the free area.
    ax.legend(loc="lower right", handlelength=1.8, borderaxespad=0.3)
    panel_letter(ax, "B", dx=-0.2, dy=1.0)

    # C. table (horizontal lines only, paper style)
    ax = fig.add_subplot(gd[1])
    ax.axis("off")
    cols = [T("group"), "n", "V½ (mV)", "k (mV)", "R²"]
    tb = ax.table(cellText=[f[:5] for f in rows_], colLabels=cols, loc="upper center", cellLoc="center",
                  colWidths=[0.26, 0.07, 0.29, 0.24, 0.14])
    tb.auto_set_font_size(False)
    tb.set_fontsize(7.2)
    tb.scale(1, 1.6)
    for (r, c_), cell in tb.get_celld().items():
        cell.set_linewidth(0.8)
        cell.set_edgecolor(BLACK)
        cell.visible_edges = "TB" if r == 0 else ("B" if r == len(rows_) else "")
        cell.set_facecolor("none")
        if r == 0:
            cell.set_text_props(fontweight="bold")
        elif c_ == 0:
            cell.set_text_props(fontweight="bold", color=rows_[r - 1][5])
    ax.set_title(T("params"), fontsize=7.8, loc="center", pad=-2)
    ax.text(-0.02, 1.08, "C", transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom", ha="left")
    if with_iv:
        gi = outer[1].subgridspec(1, 2, wspace=0.36)
        labs = {k: category_label(k, len(l)) for k, l in categories.items()}
        for j, (kind, letter) in enumerate((("step", "D"), ("tail", "E"))):
            panel_iv(fig.add_subplot(gi[j]), df_all, kind, categories, category_colors, styles, marks, labs, letter)
    save_figure(fig, folder, name)
    plt.show()

# ------------------------------------------------------------------ FIGURE 2 (paper): Control vs transfected with significance
# FIGURE 2: Control vs transfected. Statistical unit = cell. The bracket p value comes from p_two_groups with the test chosen in SIGNIFICANCE_TEST.
def figure_control_vs_herg(table, df_t, folder="results", name="Figure2_control_vs_hERG"):
    """A–B = step and tail current vs voltage (mean ± SEM) · C = maximum tail amplitude per cell with the
    significance bracket. Statistical unit = cell. Returns the result of the test (dict)."""
    figure_style()
    order_ = order_groups(table["group"].unique())
    if CONTROL_GROUP not in order_ or TRANSFECTED_GROUP not in order_:
        print(f"The groups '{CONTROL_GROUP}' and '{TRANSFECTED_GROUP}' are required (section 2e).")
        return None
    order_ = [CONTROL_GROUP, TRANSFECTED_GROUP]
    pf = cm_complete(df_t, "I_tail_pApF")
    col_amp = "I_tail_pApF" if pf else "I_tail_pA"
    amp_res = "I_tail_max_pApF" if pf else "I_tail_max_pA"
    col_g = {g: PALETTE[i] for i, g in enumerate(order_)}
    mk = {CONTROL_GROUP: "s", TRANSFECTED_GROUP: "o"}

    fig = plt.figure(figsize=(7.6, 3.5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.3, 1.3, 0.85], wspace=0.42, left=0.075, right=0.985, top=0.92, bottom=0.19)
    groups_iv = {g: table.loc[table["group"] == g, "file"].tolist() for g in order_}
    labs = {g: f"{g} (n={len(l)})" for g, l in groups_iv.items()}
    for j, (kind, letter) in enumerate((("step", "A"), ("tail", "B"))):      # step current and tail current vs voltage
        panel_iv(fig.add_subplot(gs[j]), df_t, kind, groups_iv, col_g, marks=mk, labels_=labs, letter=letter,
                 individual=True, signif_between=(order_[0], order_[1]))

    # C. maximum amplitude per cell + significance
    ax = fig.add_subplot(gs[2])
    points_by_group(ax, table, amp_res, T("amp_max_pf") if pf else T("amp_max_pa"), PALETTE, order_=order_)
    ax.spines["bottom"].set_position(("outward", 3)); ax.spines["left"].set_position(("outward", 3))
    vals = table[amp_res].dropna().astype(float)
    res = p_two_groups(table, amp_res, order_[0], order_[1], SIGNIFICANCE_TEST)
    if len(vals):
        ymax, ymin = float(vals.max()), min(0.0, float(vals.min()))
        r_ = (ymax - ymin) or 1.0
        ax.set_ylim(ymin - 0.04 * r_, ymax + 0.30 * r_)
        if np.isfinite(res["p"]):
            significance_bracket(ax, 0, 1, ymax + 0.08 * r_, f"{stars(res['p'])}  {fmt_p(res['p'])}", h=0.03 * r_, fs=8)
    panel_letter(ax, "C", dx=-0.42, dy=1.0)
    fig.text(0.085, 0.01, T("test_note").format(test=res["test_name"] or "—"), fontsize=6.3, color=GREY, ha="left", va="bottom")
    save_figure(fig, folder, name)
    plt.show()
    return res

## 5. Verification of windows, filter and baseline
This verification must be carried out before the analysis.

**5a. Windows and filter.** In the plot, the **grey** zone (baseline) must lie on a flat segment at the reference potential; the **blue** one (steady state), at the end of the pulse; and the **red** one (peak search), immediately after the onset of the tail. By default only the filtered trace is shown; the original unfiltered trace (light grey) is enabled in section 4c (`SHOW_UNFILTERED`). In the zoomed panel, the filter must reduce the noise without attenuating or delaying the tail peak. The number of sweeps and the duration shown must match the summary of the Clampex protocol used for each file.


In [ ]:
#@title 5a. Verification of windows and filter { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: (1) table with the protocol detected in ALL the files and (2) check plot of ONE file.
# Rationale: if a window is placed in the wrong location, the whole subsequent analysis is wrong; the check must be done before section 6.
# ----------------------------------------------------------------------------------------------------------
#@markdown Index of the file to check (0 = first of the list loaded in section 3)
FILE_TO_CHECK = 0  #@param {type:"integer"}

pd.set_option("display.max_colwidth", 250)
protocol_table = protocol_summary(abf_files)
display(protocol_table)
verify_windows(abf_files[FILE_TO_CHECK])


**5b. Baseline diagnostics.** The top panel shows the filtered signal before subtracting the baseline; the middle panel, after subtracting it (all traces must coincide in the grey zone); the bottom panel, the baseline value of each sweep.
This correction removes a *per-sweep offset*. It does not correct drift *within* a sweep nor the leak current at other potentials (see the note at the end of section 6).


In [ ]:
#@title 5b. Baseline diagnostics { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: shows the filtered signal before and after subtracting the baseline, and the baseline of each sweep.
# Rationale: confirms that, after subtraction, all the traces start from 0 pA at holding and that the baseline does not drift.
# ----------------------------------------------------------------------------------------------------------
#@markdown Index of the file to check
FILE_TO_CHECK_BASELINE = 0  #@param {type:"integer"}
baseline_diagnostics(abf_files[FILE_TO_CHECK_BASELINE])


## 6. Per-cell analysis
For each file the per-sweep table and a nine-panel figure are produced (black / blue / red palette):

1. **Voltage protocol** (read from the actual command of the file, with epochs A–D) and, below, the filtered **traces with the baseline of each sweep subtracted** (all start at 0 pA at holding).
2. **Aligned tails** (at the detected tail voltage).
3. **I–V curves** (end of the pulse, pulse peak and tail peak) with the E_K line.
4. Steady-state **conductance**, G = I / (V − E_K).
5. **Activation**: normalised tail and Boltzmann fit (V½, k, apparent z, errors and R²).
6. **Deactivation fit** of the largest tail (mono- vs bi-exponential).
7. Deactivation **τ** as a function of the pulse voltage.
8. **Quality control**: baseline value measured in each sweep *before* subtracting it, and its noise.
9. Text **summary**: V½, k, z, R², maximum current and quality-control status.

**Interpretation notes:**
- **Deactivation τ.** If the tail is short compared with τ (for example, a 500 ms tail at −40 mV, where hERG barely decays), τ cannot be determined and is omitted instead of reporting a meaningless value.
- **Sweeps without a defined current.** If the peak does not exceed `SNR_THRESHOLD` times the baseline noise (filtered signal) or `MIN_THRESHOLD_pA`, the maximum would correspond to noise only. In that case the tail peak is measured in a fixed window (±2 ms) around the typical peak time of the cell, and the pulse peak and kinetics are omitted.
- **Leak.** Subtracting the baseline removes a *constant* offset. A linear leak is proportional to voltage, so a residual leak persists at other potentials (pulse, tail). If the baseline is large (for example, −300 to −400 pA at −80 mV), the tails of the most negative pulses are not nulled and the activation curve is shifted. The rigorous correction is to subtract the current sensitive to a specific blocker (E-4031) or to apply a P/N protocol.


In [ ]:
#@title 6. Per-cell analysis { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: analyses each file (per-sweep table + 9 plots) and saves the CSV of each cell.
# Rationale: this is the central step: the windows, the filter, the baseline and the fits of section 4a are applied here to each cell.
# If a file fails (atypical protocol, too few sweeps) a warning is issued and the analysis continues with the others.
# ----------------------------------------------------------------------------------------------------------
os.makedirs("results", exist_ok=True)
# Dictionary {file: results}: ALL the subsequent tables are derived from it
results = {}

for path_ in abf_files:
    name = os.path.basename(path_)
    print("=" * 90)
    print(name, "| group:", group_of(name))
    try:
        res = analyze_abf(path_, CM_PF.get(name))
    except Exception as e:                      # a problematic file does not stop the others
        print(f"WARNING: {name} could not be analysed: {e}")
        continue
    results[name] = res
    res["df"].to_csv(f"results/{os.path.splitext(name)[0]}_sweeps.csv", index=False)
    bi = res["baseline_info"]
    print(f"Baseline: window '{bi['window_mask']}' ({bi['start_s']:.3f}–{bi['end_s']:.3f} s) at {bi['V_mV']:.0f} mV")
    if bi["alert"]:
        print("WARNING:", bi["alert"])
    if LEAK_CORRECTION == "holding" or OFFSET_CORRECTION != "none":
        print(f"Corrections: leak g = {res['g_leak_nS']:.2f} nS (E = {E_LEAK_mV:g} mV) · tail offset subtracted = {res['tail_offset_pA']:.0f} pA")
        if res["offset_warning"]:
            print("WARNING:", res["offset_warning"])
    display(res["df"].round(2))
    b = res["boltz"]
    if b:
        print(f"Boltzmann: V½ = {b['V50_mV']:.1f} ± {b['SE_V50']:.1f} mV · k = {b['k_mV']:.1f} ± {b['SE_k']:.1f} mV · "
              f"z = {b['z_apparent']:.2f} · R² = {b['R2']:.3f}")
    print(f"Cm estimated from the capacitive transient (approx.): {res['cm_est']:.1f} pF")
    print("Quality control:", res["qc"]["status"])
    plot_cell(path_, res)

## 7. Quality control
The fulfilment of the criteria defined in section 2d is summarised per cell. The default thresholds are indicative and must be adjusted to the criteria of the laboratory before excluding cells. In the associated study, flagged cells were not removed from the analysis; their effect was assessed by a sensitivity analysis (section 10b).


In [ ]:
#@title 7. Quality control { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: quality-control (QC) table per cell.
# Rationale: allows explicit criteria to be used to decide which recordings are reliable (leak, noise, drift, Boltzmann fit).
# ----------------------------------------------------------------------------------------------------------
qc_table = pd.DataFrame([{"file": n, "baseline_segment": r["baseline_info"]["window_mask"], "V_baseline_check_mV": r["baseline_info"]["V_mV"],
                          "I_baseline_mean_pA": r["qc"]["I_hold_mean_pA"],
                          "noise_filtered_pA": r["qc"]["noise_mean_pA"],
                          "drift_pA_per_sweep": r["qc"]["drift_pA_per_sweep"],
                          "Cm_estimated_pF": r["cm_est"], "status": r["qc"]["status"]}
                         for n, r in results.items()])
display(qc_table.round(2))

## 8. Summary across cells and descriptive statistics
A table with **one row per cell** is built, which is the appropriate statistical unit (the sweeps of one cell are *not* independent replicates).

The descriptive statistics include n, mean, standard deviation (SD), standard error of the mean (SEM), 95 % CI (Student's t), median, quartiles, range and the **Shapiro–Wilk** normality test (p > 0.05 is compatible with normality; with n < 5 its detection power is low).


In [ ]:
#@title 8. Summary across cells and descriptive statistics { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: builds the table with ONE ROW PER CELL, the table of all sweeps and the descriptive statistics per group.
# Rationale: the correct statistical unit is the cell. 'df_all' (all sweeps) is used for the curves by voltage; 'summary' (one row per cell) for the tests.
# ----------------------------------------------------------------------------------------------------------
summary = pd.DataFrame([summary_row(n, r, group_of(n)) for n, r in results.items()])
display(summary.round(2))

df_all = pd.concat([r["df"].assign(file=n, group=group_of(n)) for n, r in results.items()],
                     ignore_index=True)
# The voltage is rounded to whole mV so that the same voltage can be grouped across cells (they may differ by tenths)
df_all["V_rounded"] = df_all["V_test_mV"].round(0)

# Only the variables with at least one data point are used
VARIABLES = [v for v in ["V50_mV", "k_mV", "z_apparent", "I_tail_max_pA", "I_tail_max_pApF", "I_steady_max_pA",
                         "tau_mono_Vref_ms", "tau_fast_Vref_ms", "tau_slow_Vref_ms", "t_tail_peak_Vref_ms"]
             if summary[v].notna().sum() > 0]

desc = descriptive_stats(summary, VARIABLES)
display(desc.round(3))
if summary.groupby("group").size().max() < 5:
    print("Warning: with n < 5 cells per group the tests have low power and normality cannot be assessed well.")

## 9. Statistical comparisons between groups
They are executed only if **at least two groups** are defined (section 2e). For each variable and each pair of groups the following are obtained:

- **Difference of means with 95 % CI** (Welch) and **effect size** (Hedges' g: ~0.2 small, ~0.5 medium, ~0.8 large).
- **Welch's t-test** (does not assume equal variances) and **Mann–Whitney U** (non-parametric).
- **Levene** (equality of variances) and **Shapiro–Wilk**, which determine the suggested test: Welch if both groups are compatible with normality; Mann–Whitney otherwise.
- **Holm-corrected p** across the pairs of each variable. With three or more groups, global ANOVA and Kruskal–Wallis are also computed.

Tail amplitude and normalised activation are also compared **voltage by voltage** (Holm across voltages). No correction is applied across different variables; therefore, tail amplitude is regarded as the primary hypothesis and the remaining variables as exploratory.


In [ ]:
#@title 9. Statistical comparisons between groups { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: compares the groups pairwise (Welch, Mann–Whitney, Hedges g, Holm) and voltage by voltage.
# Rationale: answers whether Control and transfected differ, with the effect size and not only the p value. Requires ≥ 2 groups.
# ----------------------------------------------------------------------------------------------------------
n_groups = summary["group"].nunique()
if n_groups < 2:
    print("There is only one group. Define GROUPS in section 2e to enable the comparisons.")
    pair_comparisons, global_comparisons, voltage_comparisons = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
else:
    pair_comparisons, global_comparisons = compare_groups(summary, VARIABLES)
    print("Pairwise comparisons (summary across cells):")
    display(pair_comparisons.round(4))
    if len(global_comparisons):
        print("Global tests (≥ 3 groups):")
        display(global_comparisons.round(4))

    curve_var = "I_tail_pApF" if cm_complete(df_all, "I_tail_pApF") else "I_tail_pA"
    voltage_comparisons = pd.concat([compare_by_voltage(df_all, curve_var),
                              compare_by_voltage(df_all, "tail_norm")], ignore_index=True)
    print("Voltage-by-voltage comparisons:")
    display(voltage_comparisons.round(4))

### Guide to the interpretation of the tables

**The statistical unit is the cell**: `n` is the number of cells, not of sweeps.

| Column | Meaning | Interpretation |
|---|---|---|
| `mean`, `SD`, `SEM` | SD = dispersion between cells · SEM = SD/√n = uncertainty of the mean | SD describes the sample; SEM or the 95 % CI are used for comparison |
| `CI95` of the difference | Range of differences of means compatible with the data (Welch) | If it does not contain 0, it agrees with p < 0.05. A very wide CI indicates low precision (small n) |
| `Hedges_g` | Effect size in SD units | ~0.2 small · ~0.5 medium · ~0.8 large. The p value does not convey this magnitude |
| `Shapiro_p` | Compatibility of the data with a normal distribution | p < 0.05 indicates lack of compatibility (a non-parametric test is preferred). With n < 5 it almost never detects deviations |
| `p_Welch` | Compares means without assuming equal variances | Requires approximately normal data |
| `p_MannWhitney` | Compares ranks (non-parametric) | With small n there is a minimum attainable p: for 3 vs 8 it is 0.012 |
| `p_..._Holm` | p corrected for multiple comparisons (pairs or voltages) | Applicable when several comparisons are made simultaneously |
| `suggested_test` / `p_suggested_Holm` | Welch if both groups are compatible with normality; otherwise Mann–Whitney | It is the test shown in Figure 2 when `SIGNIFICANCE_TEST = "suggested"` |

**p value.** Probability of observing a difference at least as large as the one observed **if there were no real difference**. It is not the probability that the hypothesis is true nor a measure of effect size. Marks: `*` p < 0.05 · `**` p < 0.01 · `***` p < 0.001 · `ns` = not significant (it does not mean "equal").

**Considerations for the Control vs transfected comparison:**
- The variable that identifies hERG is the **tail current** (maximum amplitude and its voltage dependence). The current at the end of the pulse is not specific: hERG inactivates at positive voltages and control cells show endogenous currents.
- The **control has no activation curve**: its V½ and k values come from fitting noise (often at the fit bounds) and are not comparable with those of the transfected group.
- The control "tail current" is not null: it corresponds to residual leak (the baseline is subtracted at −80 mV, but the leak at other voltages is different). The signature of hERG is that it **grows with voltage**, whereas that of the control remains flat.
- Each variable constitutes a separate test and **no correction is applied across variables**: tail amplitude is regarded as the primary hypothesis; the rest is interpreted as exploratory.
- Selecting the "best" cells by R² is appropriate to *show* the activation, but not for hypothesis testing. If it is used (n = 6), it must be declared and accompanied by an analysis with all cells.


## 10. Mean curves and group plots
Top panel: mean ± SEM per group (optionally, individual curves as thin lines) for tail amplitude, normalised activation (with the Boltzmann fitted to the mean) and deactivation τ.
Bottom panel: V½, k and amplitude per cell (dots) with mean ± SEM.

*The Boltzmann of the mean curve is not identical to the mean of the individual V½ values; the individual values are used for the statistics (sections 8–9).*


In [ ]:
#@title 10. Mean curves and group plots { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: summary figure per group (mean ± SEM curves and one dot per cell).
# Rationale: shows the differences between groups at a glance. Note: the Boltzmann of the MEAN curve is not equal to the mean of the individual V½ values; the individual values are used for the statistics.
# ----------------------------------------------------------------------------------------------------------
groups = order_groups(summary["group"].unique())
colors = PALETTE   # the same palette (black / blue / red) in all the figures

fig, ax = plt.subplots(2, 3, figsize=(11, 6.6))
mean_curve = figure_groups(ax[0], ax[1], df_all, summary)
fig.tight_layout()
save_figure(fig, "results", "groups_summary")
plt.show()

### 10b. Comparison between all cells and cells that pass quality control (controls included in both cases)


In [ ]:
#@title 10b. All cells vs. cells that pass the QC { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: repeats the group figure with all cells and with only those that pass quality control.
# Rationale: shows whether the conclusions depend on a few poor-quality cells. Controls are ALWAYS kept (they have no activation curve that could 'fail' the QC).
# ----------------------------------------------------------------------------------------------------------
# Cells that pass quality control (the "Control" group is always kept)
filtered_cells = []
for _, row in qc_table.iterrows():
    file = row["file"]
    if group_of(file) == CONTROL_GROUP or row["status"] == "OK":
        filtered_cells.append(file)

df_filtered = df_all[df_all["file"].isin(filtered_cells)].copy()
summary_filtered = summary[summary["file"].isin(filtered_cells)].copy()

groups = order_groups(summary["group"].unique())
colors = PALETTE

fig, ax = plt.subplots(4, 3, figsize=(11, 13))
figure_groups(ax[0], ax[1], df_all, summary, prefix="All · ")
figure_groups(ax[2], ax[3], df_filtered, summary_filtered, prefix="QC · ")
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.suptitle('Rows 1–2, "All": ' + T("all_cells") + '   |   Rows 3–4, "QC": ' + T("filtered"), fontsize=10, fontweight="bold")
save_figure(fig, "results", "groups_all_vs_QC")
plt.show()

In [ ]:
#@title 10c. Quality categories (best, middle, worst) and best recordings { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: ranks the transfected cells by Boltzmann R² and splits them into best / middle / worst; chooses the best recording of each group for panel A.
# Rationale: allows the activation to be shown with representative cells. Selection by R² can correlate with current amplitude, so if it is used in the statistics it must be declared and accompanied by a sensitivity analysis with all cells.
# ----------------------------------------------------------------------------------------------------------
# 10c. Categories by quality of the Boltzmann fit — transfected cells ONLY
#      (the control has no activation curve to fit: its R² is meaningless).
#      Figure 1 (with table): 3 BEST, 3 MIDDLE and 2 WORST (by R²).
#      The figure without table uses the cells defined by CELLS_FOR_FIGURE: "best_and_middle" (3 best and 3 middle, n = 6)
#      or "all" the transfected ones.
#@markdown Number of cells per category (best + middle = n of Figure 1 and of the "best_and_middle" option)
N_BEST = 3  #@param {type:"integer"}
N_MIDDLE = 3   #@param {type:"integer"}
N_WORST = 2   #@param {type:"integer"}
# Only transfected cells with a valid Boltzmann fit (the control has no activation curve: its R² is meaningless), sorted from highest to lowest R²
transfected = (summary[summary["group"] != CONTROL_GROUP].dropna(subset=["R2_boltzmann"])
                 .sort_values("R2_boltzmann", ascending=False).reset_index(drop=True))
categories = None               # 3 best + 3 middle (n = 6)
categories_with_worst = None    # + the 2 worst: figure with table
if len(transfected) < N_BEST + N_MIDDLE:
    print(f"At least {N_BEST + N_MIDDLE} transfected cells with a valid Boltzmann fit are required. There are {len(transfected)}.")
    print("Figure 1 will use the groups defined in GROUPS.")
else:
    categories = {"best": transfected.iloc[:N_BEST]["file"].tolist(),
                  "mid": transfected.iloc[N_BEST:N_BEST + N_MIDDLE]["file"].tolist()}
    worst = transfected.iloc[max(N_BEST + N_MIDDLE, len(transfected) - N_WORST):]["file"].tolist()
    categories_with_worst = dict(categories, **({"worst": worst} if worst else {}))
    display(pd.DataFrame([(category_label(k, len(v)), a) for k, v in categories_with_worst.items() for a in v],
                         columns=["category", "abf_file"]).merge(transfected[["file", "R2_boltzmann", "QC"]], left_on="abf_file", right_on="file").drop(columns="file").round(4))

# --- The BEST recording of each group (the one drawn in Figure 1A) ---
# Transfected: highest Boltzmann R² among those that pass the QC (if none passes, among all).
# Control: the cleanest (lowest noise) among those that meet the QC noise and drift criteria (if none, among all).
# Best transfected: highest R² among those that pass the QC. Best control: the cleanest (lowest noise) among those meeting noise and drift.
best_herg = best_control = None
if len(transfected):
    passing = transfected[transfected["QC"] == "OK"]
    best_herg = (passing if len(passing) else transfected).iloc[0]["file"]
    row_ = transfected[transfected["file"] == best_herg].iloc[0]
    print(f"Best {TRANSFECTED_GROUP} recording: {best_herg} · R² = {row_['R2_boltzmann']:.3f} · max tail = {row_['I_tail_max_pA']:.0f} pA · QC: {row_['QC']}")
ctrl = summary[summary["group"] == CONTROL_GROUP]
if len(ctrl):
    clean = ctrl[(ctrl["noise_pA"] <= QC_MAX_NOISE_pA) & (ctrl["drift_pA_per_sweep"].abs() <= QC_MAX_DRIFT_pA_PER_SWEEP)]
    best_control = (clean if len(clean) else ctrl).sort_values("noise_pA").iloc[0]["file"]
    row_ = ctrl[ctrl["file"] == best_control].iloc[0]
    print(f"Best {CONTROL_GROUP} recording: {best_control} · noise = {row_['noise_pA']:.1f} pA · drift = {row_['drift_pA_per_sweep']:.1f} pA/sweep")

In [ ]:
#@title 10c-2. Figure 1 (with table) { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: draws Figure 1 with table: A = protocol and traces (transfected + control, same scale), B = activation by category, C = Boltzmann parameters, D–E = step and tail vs voltage.
# Rationale: it is the main figure of the article; table C gives V½ and k (mean ± SEM) of each category.
# ----------------------------------------------------------------------------------------------------------
# Figure 1 (article): a single figure, half and half
#   A (left): voltage protocol + traces of the BEST transfected recording and, below, those of the CONTROL (same pA and ms scale)
#   B (right, top): activation curves of the 3 best, the 3 middle and the 2 worst (n = 8) · legend at bottom right
#   C (right, bottom): Boltzmann fit parameters
# Time window and calibration bars: section 4c (FIG_T_START_S, FIG_T_END_S, SCALE_PA, SCALE_MS).
#@markdown File for panel A. Empty = automatic (the best transfected and the cleanest control, see 10c)
HERG_CELL = ""      #@param {type:"string"}
CONTROL_CELL = ""   #@param {type:"string"}
#@markdown Include the current-vs-voltage panels (D–E in the figure with table; C in the figure without table)
FIG_WITH_IV = True        #@param {type:"boolean"}
#@markdown Panel A with the transfected AND the control cell (False = transfected only)
FIG_WITH_CONTROL = True   #@param {type:"boolean"}

def _abf_path(name):
    name = name if name.lower().endswith(".abf") else name + ".abf"
    return os.path.join("data", name)

n_herg = HERG_CELL.strip() or best_herg
n_ctrl = CONTROL_CELL.strip() or best_control
herg_path = _abf_path(n_herg) if n_herg else abf_files[0]
ctrl_path = _abf_path(n_ctrl) if (FIG_WITH_CONTROL and n_ctrl) else None
print(f"Panel A → {TRANSFECTED_GROUP}: {os.path.basename(herg_path)}" + (f" · {CONTROL_GROUP}: {os.path.basename(ctrl_path)}" if ctrl_path else ""))

figure_main(herg_path, categories_with_worst or categories, control_path=ctrl_path, with_iv=FIG_WITH_IV)

In [ ]:
#@title Function: normalised step and Figure 1 without table { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: defines Figure 1 without table (A traces · B combined activation · C normalised step).
# Rationale: in B each cell is normalised to ITS maximum, the cells are averaged at each voltage and the bars are the dispersion BETWEEN cells; V½ and k in the legend are mean ± dispersion of the individual fits.
# ----------------------------------------------------------------------------------------------------------
# Step current normalised to the maximum of EACH cell; normalised counterpart of the tail current
def panel_normalized_step(ax, abf_files, color=None, dispersion="SD", letter=None):
    """Step current (at the end of the pulse) normalised to the maximum of EACH cell; mean ± dispersion across cells
    (voltages with ≥ 2 cells are plotted). It is the normalised counterpart of the tail current of panel B."""
    color = BLUE if color is None else color
    uses_sd = str(dispersion).upper() in ("SD", "SD")
    d = df_all[df_all["file"].isin(abf_files)].copy()
    d["step_norm"] = d.groupby("file")["I_steady_pA"].transform(lambda x: x / x.max() if x.max() > 0 else np.nan)
    agg = d.groupby("V_rounded")["step_norm"].agg(["mean", "std", "sem", "count"]).dropna(subset=["mean"])
    agg = agg[agg["count"] >= 2]
    err = (agg["std"] if uses_sd else agg["sem"]).fillna(0)
    ax.axhline(0, color=GREY, lw=0.6, ls=(0, (4, 3)), zorder=0)
    ax.errorbar(agg.index, agg["mean"], yerr=err, marker="o", ms=4.2, color=color, mfc="white", mew=1.2, lw=1.3,
                capsize=2, elinewidth=0.9, zorder=3)
    ax.set(xlabel=T("v_mem"), ylabel=T("step_norm"), ylim=(-0.06, 1.15))
    ax.spines["bottom"].set_position(("outward", 3)); ax.spines["left"].set_position(("outward", 3))
    if letter:
        panel_letter(ax, letter, dx=-0.2, dy=1.0)

# Figure 1 with two or three panels, without table
def figure_main_no_table(path_, categories=None, table=None, folder="results", name="Figure1_hERG_no_table",
                               control_path=None, dispersion="SD", with_iv=True):
    """
    Figure 1 with two panels (A and B), without table:
      A (left): voltage protocol + traces of the best transfected recording (and, if control_path is given, those of the control).
      B (right, top): COMBINED activation curve of all the cells in `categories` (e.g. 3 best + 3 middle = n = 6):
                   each cell is normalised to its own maximum, the cells are averaged at each voltage and the bars are the
                   dispersion across cells (dispersion = "SD" standard deviation · "SEM" standard error).
                   The legend (bottom right) gives V½ and k as mean ± dispersion of the fits of EACH cell.
      C (right, bottom, if with_iv=True): step current normalised to the maximum of each cell, mean ± dispersion.
    """
    figure_style()
    table = summary if table is None else table
    if categories is None:
        combined_files = table.loc[table["group"] != CONTROL_GROUP, "file"].tolist()
    else:
        combined_files = [a for items in categories.values() for a in items]
    n_total = len(combined_files)
    with_control = control_path is not None
    uses_sd = str(dispersion).upper() in ("SD", "SD")
    disp_label = "SD" if uses_sd else "SEM"

    if with_iv:      # right: B (normalised tail) on top and C (normalised step) below
        fig = plt.figure(figsize=(7.6, 5.6 if with_control else 4.6))
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.28, left=0.08, right=0.97, top=0.95, bottom=0.10)
    else:
        fig = plt.figure(figsize=(7.6, 4.8 if with_control else 3.5))
        gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.28, left=0.08, right=0.97, top=0.92, bottom=0.13)

    # Panel A: traces
    if with_control:
        panel_protocol_traces_two(fig, gs[0], _prepare_cell(path_), _prepare_cell(control_path), FIG_T_START_S, FIG_T_END_S, "A")
    else:
        panel_protocol_traces(fig, gs[0], pyabf.ABF(path_), "A", FIG_T_START_S, FIG_T_END_S, bars=True, heights=(1.0, 2.1), hspace=0.02)

    # Panel B: combined curve = mean of the normalised curves of each cell, with its dispersion at each voltage
    gd = gs[1].subgridspec(2, 1, hspace=0.50) if with_iv else None
    ax_b = fig.add_subplot(gd[0] if with_iv else gs[1])
    ax_b.axhline(0.5, color=LIGHT_GREY, lw=0.7, ls=":", zorder=0)
    dfc = df_all[df_all["file"].isin(combined_files)]
    agg = dfc.groupby("V_rounded")["tail_norm"].agg(["mean", "std", "sem", "count"]).dropna(subset=["mean"])
    err = (agg["std"] if uses_sd else agg["sem"]).fillna(0)

    # V½ and k: mean ± dispersion of the individual fits (one row = one cell)
    pars = table[table["file"].isin(combined_files)][["file", "V50_mV", "k_mV"]].dropna()
    def m_disp(x):
        x = x.astype(float)
        if len(x) < 2:
            return x.mean(), np.nan
        d = x.std(ddof=1) if uses_sd else x.std(ddof=1) / np.sqrt(len(x))
        return x.mean(), d
    def txt(x, unit="mV"):
        m, d = m_disp(x)
        return f"{_fmt_num(m, 1)}" + (f" ± {_fmt_num(d, 1)}" if np.isfinite(d) else "") + f" {unit}"
    if len(pars):
        legend_label = f"{TRANSFECTED_GROUP} (n = {n_total})\nV½ = {txt(pars['V50_mV'])}\nk = {txt(pars['k_mV'])}"
    else:
        legend_label = f"{TRANSFECTED_GROUP} (n = {n_total})"

    # The line is the Boltzmann fitted to the MEAN curve; V½ and k in the legend come from the fits of each cell
    bm = fit_boltzmann(agg.index.values, agg["mean"].values)       # the line is the fit to the mean curve
    if bm:
        vv = np.linspace(agg.index.min() - 5, agg.index.max() + 5, 200)
        ax_b.plot(vv, boltzmann(vv, bm["A"], bm["V50_mV"], bm["k_mV"]), color=BLUE, ls="-", lw=1.5)
    ax_b.errorbar(agg.index, agg["mean"], yerr=err, fmt="o", ms=4.2, color=BLUE, mfc="white", mew=1.2,
                  capsize=2, elinewidth=0.9, label=legend_label, zorder=3)
    ax_b.set(xlabel=T("v_mem"), ylabel=T("i_norm"), ylim=(-0.06, 1.15))
    ax_b.spines["bottom"].set_position(("outward", 3))
    ax_b.spines["left"].set_position(("outward", 3))
    ax_b.legend(loc="lower right", handlelength=1.8, borderaxespad=0.3, fontsize=7.2, labelspacing=0.8)
    panel_letter(ax_b, "B", dx=-0.2, dy=1.0)
    if with_iv:
        panel_normalized_step(fig.add_subplot(gd[1]), combined_files, BLUE, dispersion, "C")
    save_figure(fig, folder, name)
    plt.show()

    # Only printed (not drawn): which cells were included and what was computed
    global combined_parameters
    combined_parameters = pars.reset_index(drop=True)
    if len(pars):
        # Warning: a fit stuck at the bound (k = 60 mV or V½ = ±100 mV) is not well determined and inflates the dispersion
        at_limit = pars[(pars["k_mV"] >= 59.5) | (pars["k_mV"] <= 1.05) | (pars["V50_mV"].abs() >= 99.5)]
        if len(at_limit):
            print("Warning (console only): the individual fit of " + ", ".join(at_limit["file"]) +
                  " reached the allowed limit (k = 60 mV or V½ = ±100 mV): its V½/k is not well determined and inflates the dispersion in the legend.")
    print(f"Combined curve: n = {n_total} cells · {', '.join(combined_files)}")
    if len(pars):
        print(f"  V½ = {txt(pars['V50_mV'])} · k = {txt(pars['k_mV'])}   (mean ± {disp_label} of the individual fits, n = {len(pars)})")
    print(f"  Error bars of the curve: {disp_label} between cells at each voltage (minimum n per voltage = {int(agg['count'].min())})")

In [ ]:
#@title 10c-3. Figure 1 (without table) { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: draws Figure 1 without table with the chosen cells and records which cells were included (DESC_FIG1).
# Rationale: it is the clean version for the article; using n = 6 or n = 8 is a decision that must be declared in the text.
# ----------------------------------------------------------------------------------------------------------
# Figure 1 without table: A traces · B normalised tail · C normalised step (FIG_WITH_IV = False removes C)
#@markdown Error bars: SD = standard deviation across cells · SEM = standard error of the mean
DISPERSION = "SD"  #@param ["SD", "SEM"]
#@markdown Cells in panel B: **all** the transfected ones (e.g. n = 8) or only **best_and_middle** (n = 6, from section 10c)
CELLS_FOR_FIGURE = "best_and_middle"  #@param ["all", "best_and_middle"]
use_cells = None if (CELLS_FOR_FIGURE == "all" or categories is None) else categories
n_fig1 = (len(summary[summary["group"] != CONTROL_GROUP]) if use_cells is None else sum(len(l) for l in use_cells.values()))
DESC_FIG1 = (f"all {TRANSFECTED_GROUP} cells (n = {n_fig1})" if use_cells is None
             else f"the best and middle cells by Boltzmann R² (n = {n_fig1})")
figure_main_no_table(herg_path, use_cells, control_path=ctrl_path, dispersion=DISPERSION, with_iv=FIG_WITH_IV)

### 10d. Figure 2 — Control vs transfected, with significance
Tail current amplitude versus voltage (A, B) and maximum amplitude per cell with the significance bracket (C). Each cell is one dot. The statistical test is selected in section 4c (`SIGNIFICANCE_TEST`). The transfected cells included are defined by `CELLS_FOR_STATISTICS`: `all` (e.g. n = 8) or `best_and_middle` (3 best and 3 middle by Boltzmann R², n = 6); controls are always included in full. Selection by R² is appropriate to *show* the activation, but it can bias the test; if used, it must be declared and accompanied by a sensitivity analysis with all cells.


In [ ]:
#@title 10d. Figure 2 · Control vs transfected { display-mode: "form" }

# ----------------------------------------------------------------------------------------------------------
# Description: compares the Control group with the transfected group (tail amplitude vs voltage and maximum
#           amplitude per cell) and draws the significance bracket of Figure 2.
# Rationale: the tail current is the one that identifies hERG. The statistical unit is the CELL (one row per
#          cell), not the sweep, because sweeps from the same cell are not independent replicates.
# ----------------------------------------------------------------------------------------------------------
#@markdown **Which transfected cells enter the statistics and the figure?**
#@markdown - **all**: all transfected cells (e.g. n = 8), without selection.
#@markdown - **best_and_middle**: only the 3 best + 3 middle by Boltzmann R² (n = 6, the same as Figure 1; requires section 10c).
#@markdown Controls are always included in full.
CELLS_FOR_STATISTICS = "best_and_middle"  #@param ["all", "best_and_middle"]
#@markdown **COMPARE_WITH**: `all` = not filtered by quality · `QC` = only those that pass quality control (controls always included; requires section 10b).
COMPARE_WITH = "all"  #@param ["all", "QC"]

# 1) Starting point: all cells, or only those that pass the QC
if COMPARE_WITH == "QC":
    table_cmp, df_cmp = summary_filtered, df_filtered
else:
    table_cmp, df_cmp = summary, df_all

# 2) Choose n = 8 (all) or n = 6 (best + middle). Both the table by cell AND the curves by voltage are filtered,
#    so that the curve panel and the dot panel use exactly the same cells.
cells_desc_fig2 = "all transfected cells"
if CELLS_FOR_STATISTICS == "best_and_middle":
    if categories is None:
        print("WARNING: there are no quality categories (section 10c needs ≥ 6 transfected cells with a valid Boltzmann fit): all are used.")
    else:
        chosen = [a for items in categories.values() for a in items]        # 3 best + 3 middle
        keep = (table_cmp["group"] == CONTROL_GROUP) | table_cmp["file"].isin(chosen)
        table_cmp = table_cmp[keep].copy()
        df_cmp = df_cmp[df_cmp["file"].isin(table_cmp["file"])].copy()
        cells_desc_fig2 = "the best and middle cells by Boltzmann R²"
n_ctrl_f2 = int((table_cmp["group"] == CONTROL_GROUP).sum())
n_herg_f2 = int((table_cmp["group"] == TRANSFECTED_GROUP).sum())
print(f"Statistics of Figure 2: {CONTROL_GROUP} n = {n_ctrl_f2} vs {TRANSFECTED_GROUP} n = {n_herg_f2} ({cells_desc_fig2}"
      + (", only cells that pass the QC" if COMPARE_WITH == "QC" else "") + ")\n")

# 3) Unit: pA/pF only if ALL the cells compared have Cm (otherwise pA; so units are not mixed)
has_pf = cm_complete(table_cmp, "I_tail_max_pApF")
amp_var, amp_unit = ("I_tail_max_pApF", "pA/pF") if has_pf else ("I_tail_max_pA", "pA")

# 4) Plain-language summary (mean ± SEM, difference with CI95 %, effect size, p) and the figure
summary_two_groups(table_cmp, amp_var, CONTROL_GROUP, TRANSFECTED_GROUP, SIGNIFICANCE_TEST, amp_unit)
print()
res_fig2 = figure_control_vs_herg(table_cmp, df_cmp)
if res_fig2:      # records which cells were used (exported to the Excel file)
    res_fig2["cells"] = cells_desc_fig2
    res_fig2["n_control"], res_fig2["n_transfected"] = n_ctrl_f2, n_herg_f2


### 10e. Dependence of V½ and k on the constant (offset) tail current
All cells, controls included, show a "tail" at −70 mV in which hERG should not be activated. That constant current prevents the normalised curve from falling to zero and shifts V½ and k. Each cell is fitted in three ways:
**(1)** no correction · **(2)** subtracting the −70 mV tail · **(3)** Boltzmann with free offset (uses all the points, with greater uncertainty).
If the three methods give similar V½ and k values, the result is robust; otherwise, that V½ is not well determined with the protocol used. The rigorous correction is to subtract the current sensitive to a specific blocker (E-4031).


In [ ]:
#@title 10e. Sensitivity of V½ and k to the constant tail current { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: fits each transfected cell in three ways (no correction, subtracting the −70 mV tail, and Boltzmann with free offset).
# Rationale: if the three methods give similar V½ and k the result is robust; if not, V½ is not well determined with this protocol. It does NOT modify anything in the analysis.
# ----------------------------------------------------------------------------------------------------------
# 10e. Sensitivity of V½ and k to the constant tail current (does not modify anything in the analysis)
# Boltzmann with an extra parameter i0 (constant background current): uses all the points, but with more uncertainty
def _boltz_free_offset(v, ic):
    def f(v_, i0, a, v50, k):
        return i0 + a / (1 + np.exp((v50 - v_) / k))
    try:
        p, cv = curve_fit(f, v, ic, p0=[ic.min(), ic.max() - ic.min(), -30, 15],
                          bounds=([0, 0, -100, 1], [ic.max(), 3 * ic.max(), 100, 60]), maxfev=40000)
        return p, np.sqrt(np.diag(cv))
    except Exception:
        return None, None

sweep_rows = []
for n in sorted(results):
    if group_of(n) == CONTROL_GROUP:
        continue
    d_ = results[n]["df"].sort_values("V_test_mV")
    v_ = d_["V_test_mV"].values
    ic_ = (d_["I_tail_raw_pA"] if "I_tail_raw_pA" in d_ else d_["I_tail_pA"]).values.astype(float)
    b1 = fit_boltzmann(v_, ic_ / ic_.max())
    b2 = fit_boltzmann(v_, (ic_ - ic_[0]) / (ic_.max() - ic_[0])) if ic_.max() > ic_[0] else None
    p3, se3 = _boltz_free_offset(v_, ic_)
    sweep_rows.append({"file": n, "I_tail_lowest_V_pA": ic_[0],
                    "V½_uncorrected": b1["V50_mV"] if b1 else np.nan, "k_uncorrected": b1["k_mV"] if b1 else np.nan,
                    "V½_minus_Vmin": b2["V50_mV"] if b2 else np.nan, "k_minus_Vmin": b2["k_mV"] if b2 else np.nan,
                    "V½_free_offset": p3[2] if p3 is not None else np.nan, "SE_V½_free": se3[2] if se3 is not None else np.nan,
                    "k_free_offset": p3[3] if p3 is not None else np.nan})
sens = pd.DataFrame(sweep_rows)
sens["V½_range_methods_mV"] = sens[["V½_uncorrected", "V½_minus_Vmin", "V½_free_offset"]].max(axis=1) - sens[["V½_uncorrected", "V½_minus_Vmin", "V½_free_offset"]].min(axis=1)
display(sens.round(1))
print("Cells whose V½ changes by more than 15 mV depending on the method (non-robust V½):",
      ", ".join(sens.loc[sens["V½_range_methods_mV"] > 15, "file"]) or "none")

cell_sets = {"all transfected cells": sens["file"].tolist()}
if categories:
    cell_sets["those of Figure 1 (categories)"] = [a for l in categories.values() for a in l]
print("\nCombined V½ and k (mean ± SD across cells):")
for nm, items in cell_sets.items():
    s_ = sens[sens["file"].isin(items)]
    for met, cv_, ck_ in [("no correction", "V½_uncorrected", "k_uncorrected"), ("subtract V-min tail", "V½_minus_Vmin", "k_minus_Vmin"),
                          ("free offset", "V½_free_offset", "k_free_offset")]:
        print(f"  {nm} (n = {len(s_)}) · {met:<20} V½ = {s_[cv_].mean():6.1f} ± {s_[cv_].std(ddof=1):4.1f} mV · k = {s_[ck_].mean():5.1f} ± {s_[ck_].std(ddof=1):4.1f} mV")

## 11. Correlations and outliers
- **Spearman correlations** between parameters (they do not assume normality; they require n ≥ 5 cells per group; if no group reaches that number, all cells are pooled and a correlation may only reflect the difference between groups). They allow assessing, for example, whether amplitude depends on cell size or whether V½ varies with amplitude (possible *run-down* or series-resistance effect).
- **Outliers** according to the modified z-score (median and MAD): a value with |z| > 3.5 is flagged for review; it is **not** removed automatically.


In [ ]:
#@title 11. Correlations and outliers { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: Spearman correlations between parameters and list of outlying values.
# Rationale: detects whether amplitude depends on cell size (Cm), whether V½ moves with amplitude (run-down or series resistance) or whether there are unusual cells to review. Outliers are flagged, not deleted.
# ----------------------------------------------------------------------------------------------------------
corr_vars = [v for v in VARIABLES + ["Cm_used_pF", "Cm_estimated_pF", "I_hold_mean_pA"] if v in summary and summary[v].notna().sum() >= 5]
corr = correlations(summary, corr_vars)
if len(corr):
    display(corr.round(4).head(20))
else:
    print("At least 5 cells with data are required to compute correlations.")

outl = outliers(summary, VARIABLES)
if len(outl):
    display(outl.round(3))
else:
    print("No outliers detected (or n < 4 per group).")

## 12. Export of tables (source data)
An Excel workbook with all the tables and the corresponding CSV files are generated in the `results/` folder.


In [ ]:
#@title 12. Export of tables (source data) { display-mode: "form" }
# ----------------------------------------------------------------------------------------------------------
# Description: saves all the tables in an Excel workbook (one sheet per table) and as CSV files in the 'results' folder.
# Rationale: allows the reported numerical values (source data of the figures and tables) to be verified and reused.
# ----------------------------------------------------------------------------------------------------------
os.makedirs("results", exist_ok=True)
summary.to_csv("results/cell_summary.csv", index=False)

# Also export the individual statistics tables as CSV files
if 'desc' in globals() and desc is not None and not desc.empty:
    desc.to_csv("results/descriptive_statistics.csv", index=False)
if 'pair_comparisons' in globals() and pair_comparisons is not None and not pair_comparisons.empty:
    pair_comparisons.to_csv("results/group_comparisons_pairwise.csv", index=False)
if 'voltage_comparisons' in globals() and voltage_comparisons is not None and not voltage_comparisons.empty:
    voltage_comparisons.to_csv("results/voltage_by_voltage_comparisons.csv", index=False)

sheets = {"cell_summary": summary, "descriptive_stats": desc, "pairwise_comparisons": pair_comparisons,
         "global_tests": global_comparisons, "voltage_comparisons": voltage_comparisons,
         "mean_curve": mean_curve, "all_sweeps": df_all, "correlations": corr,
         "outliers": outl, "quality_control": qc_table}
if "res_fig2" in globals() and res_fig2:
    sheets["control_vs_transfected"] = pd.DataFrame([res_fig2])
if "combined_parameters" in globals():
    sheets["combined_curve_parameters"] = combined_parameters
try:
    with pd.ExcelWriter("results/hERG_results.xlsx") as xw:
        for sheet_title, table in sheets.items():
            if table is not None and len(table):
                table.to_excel(xw, sheet_name=sheet_title[:31], index=False)
except ImportError:
    print("openpyxl is missing; the Excel file is skipped (the CSV files are still saved).")
